<p class="lead">This notebook expands the fifth unit of the course, in R. The lecture slides are a map; this text is the territory. Units 2–4 trained models on data whose answer was known: every aspirate carried a biopsy result, every image a diagnosis, and the quality of a model was the agreement between its predictions and those labels. This unit removes the labels. What remains is a data matrix and three questions that can be asked of it without any answer key: <em>do the cases fall into groups?</em> (clustering — k-means and hierarchical clustering), <em>can the many measured features be summarised by a few?</em> (dimensionality reduction — principal component analysis), and <em>can mixed signals be separated into their sources?</em> (blind source separation — independent component analysis). Each method is derived, traced by hand on a paper-sized example, confirmed in R, exercised on the breast-cancer data that recur throughout the course, and pushed until it fails, because knowing <em>when</em> a method fails is the difference between using it and being used by it. Laboratory 3 applies these four methods; Workshop 3 is a questionnaire on them; this text is the preparation for both.</p>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Define unsupervised learning and distinguish clustering, dimensionality reduction and density estimation; explain what a clustering result can and cannot tell you in the absence of labels.</li>
<li>State the k-means objective (within-cluster sum of squares), execute Lloyd's algorithm by hand on a small dataset, and explain why it converges to a local, not a global, minimum.</li>
<li>Describe the main variants of k-means (fuzzy c-means, k-medoids, mini-batch, k-means++, spherical) and say when each is appropriate.</li>
<li>Choose the number of clusters with the elbow method, the silhouette and the gap statistic, and name the situations in which k-means fails (unequal sizes, non-convex shapes, unscaled features).</li>
<li>Execute agglomerative hierarchical clustering by hand, state the single, complete, average and Ward linkage rules, read a dendrogram, cut it, and compute the cophenetic correlation.</li>
<li>Derive principal component analysis as variance maximisation and as minimum reconstruction error; compute the principal components of a small dataset by hand from the covariance matrix; interpret a scree plot, loadings and a biplot; explain why standardisation comes first.</li>
<li>Explain independent component analysis as blind source separation driven by non-Gaussianity, implement FastICA, and use it to remove an artefact from a multichannel physiological recording.</li>
<li>Evaluate an unsupervised result with internal and external indices and recognise circular validation.</li>
</ul></div>

<div class="box bridge"><h4>How to read this notebook</h4>
<p>Each code chunk is short and commented. Run it, read its output, then read the paragraph that interprets it. Blue boxes carry key ideas, orange boxes carry common misconceptions, and purple boxes connect this unit to other units or to the bioinformatics course (BGM324). The notebook uses only base R and the packages <code>mlbench</code>, <code>cluster</code>, <code>MASS</code> and <code>ggplot2</code>, all pre-installed in this environment; in a fresh Google Colab R runtime a student runs <code>install.packages(c("mlbench","cluster"))</code> once. No data is downloaded. Three algorithms are traced by hand (k-means in Section 2.4, agglomerative clustering in Section 3.3, PCA in Section 4.2) before code confirms them, and two algorithms that have no base-R function — <strong>FastICA</strong> (Section 5) and <strong>spectral clustering</strong> (Section 6) — are implemented from scratch in a dozen lines each, exactly as the Python edition wrote its own. The vocabulary of Supplement S1 — train/test split, cross-validation, accuracy, leakage — is assumed and used without re-derivation.</p></div>

<div class="box warn"><h4>A note on the dataset, relative to the Python edition</h4><p>The Python edition of this unit used scikit-learn's 30-feature Wisconsin Diagnostic Breast Cancer data. R's <code>mlbench</code> ships the older <strong>9-feature</strong> Wisconsin breast-cancer set (the nine cytological scores, each on a 1–10 ordinal scale). Because all nine features already share one scale, several numbers differ from the Python edition and, in particular, standardisation changes the result far less than it did there — a contrast this unit uses rather than hides (Sections 2.8 and 4.3). Where the Python edition relied on datasets with no R equivalent — the handwritten digits and the wine chemistry table — this edition <em>simulates</em> an equivalent and says so at the point of use.</p></div>



In [ ]:
# Libraries used throughout. In a fresh Colab R runtime: install.packages(c("mlbench","cluster")).
suppressPackageStartupMessages({ library(mlbench); library(cluster); library(MASS) })
set.seed(0)                                  # every number quoted below is reproducible
BLUE <- "#2b5d9b"; ORANGE <- "#d2691e"; GREEN <- "#3a7d44"; GREY <- "#7f7f7f"; PURPLE <- "#7b4f9d"
PALETTE <- c(BLUE, ORANGE, GREEN, PURPLE, "#b5a000", "#9b2b2b", GREY)

# The adjusted Rand index (Section 2.8), used in many later cells, defined once here.
adjusted_rand <- function(a, b) {
  tab <- table(a, b); n <- sum(tab); sc <- function(x) sum(choose(x, 2))
  ai <- rowSums(tab); bj <- colSums(tab)
  idx <- sc(as.vector(tab)); exp_idx <- sc(ai) * sc(bj) / choose(n, 2); max_idx <- (sc(ai) + sc(bj)) / 2
  (idx - exp_idx) / (max_idx - exp_idx)
}
cat("R", as.character(getRversion()), "| mlbench, cluster, MASS loaded | ARI helper ready\n")



The printout confirms the environment. `set.seed(0)` fixes every random draw, so the figures and the numbers in the prose are exactly those the code produces; the `adjusted_rand` helper is defined once and reused.

## 1. Learning without labels

### 1.1 What changes when the labels are removed

In every unit so far, the training sample was a pair \( (\mathbf{X}, \mathbf{y}) \): a data matrix of \(n\) cases described by \(p\) features, and a vector of \(n\) known answers. The learning algorithm adjusted a model until its outputs agreed with \(\mathbf{y}\), and Supplement S1 explained how to measure that agreement on cases the model had not seen. This is <strong>supervised</strong> learning, named after the teacher who supplies the answers.

<strong>Unsupervised learning</strong> starts from \(\mathbf{X}\) alone. There is no \(\mathbf{y}\), hence no error to minimise against a truth, hence no accuracy, sensitivity or ROC curve. What the algorithm has instead is the <em>structure</em> of \(\mathbf{X}\): which cases resemble which, which features vary together, which directions in feature space carry most of the variation. The three main tasks are:

| Task | Question asked of \(\mathbf{X}\) | Output | Methods in this unit |
|---|---|---|---|
| **Clustering** | Do the \(n\) cases fall into groups of mutually similar cases? | A partition of the rows (a label per case, invented by the algorithm) | k-means (Section 2), hierarchical clustering (Section 3), Gaussian mixtures, DBSCAN, spectral clustering (Section 6) |
| **Dimensionality reduction** | Can the \(p\) features be replaced by \(q \ll p\) new ones that keep most of the information? | A transformation of the columns (new coordinates per case) | PCA (Section 4), ICA (Section 5); t-SNE and UMAP for visualisation (Section 4.7) |
| **Density estimation** | What is the probability distribution that generated the rows? | A model \(p(\mathbf{x})\) that assigns a probability density to any point | Gaussian mixture models (Section 6); the histograms and kernel density estimates of Unit 3 |

The three are related — a mixture model is a density estimate whose components are clusters; clustering is the most aggressive dimensionality reduction, from \(p\) features to one categorical variable — and all look for a <em>compact description</em> of the data: a few centroids, a few axes, a few components. The compactness is what makes the description useful and what makes it lossy.

### 1.2 What unsupervised learning can and cannot tell you

An unsupervised method always returns an answer. Asked for three clusters, k-means returns three clusters whether the data have three groups, two, or none at all; asked for two components, PCA returns two whether they explain 95 % or 15 % of the variance. This is the central methodological difficulty of the unit: <strong>the output cannot be validated against a truth, because no truth was supplied</strong>. What can be done, and what Section 7 develops, is to measure internal properties (are the clusters compact and separated? how much variance do the components retain?), to measure stability (does the result change when the sample is perturbed?), and — when labels exist but were hidden, as in every example of this notebook — to compare the result with them <em>after the fact</em>, as an external check that was never available to the algorithm.

The last point deserves care. The breast-cancer data<sup class="c"><a href="#r1">[1]</a></sup> carry a biopsy label, and this notebook will hide it, cluster the nuclear features, and then ask whether the clusters coincide with the diagnosis. When they do, the temptation is to say that k-means "diagnosed" the tumours. It did not: it found that malignant and benign nuclei differ enough in size and shape that the two most compact groups in the data happen to separate them. Had the dominant source of variation been, say, the laboratory that prepared the slide, k-means would have found laboratories instead, with equal confidence. Unit 2 made this correction once and it is repeated here because the error is common in the applied literature: <em>clustering discovers structure; it does not discover the structure you were hoping for</em>.<sup class="c"><a href="#r2">[2]</a></sup>

Why, then, learn without labels? Because labels are expensive (a biopsy, a pathologist's hour, a ten-year follow-up), because they may not exist yet (a disease whose subtypes are unknown — Section 8), because the question may be exploratory, and because preprocessing a supervised problem — reducing correlated features to a few uncorrelated ones — is itself unsupervised. Clustering alone has a literature of thousands of algorithms;<sup class="c"><a href="#r3">[3]</a></sup> this unit presents the two families that account for most of its use, and the two decompositions that account for most of dimensionality reduction.

### 1.3 The running example, with the labels hidden

The chunk below loads the Wisconsin breast-cancer data from <code>mlbench</code>,<sup class="c"><a href="#r4">[4]</a></sup> drops the identifier and the incomplete records, converts the nine ordinal scores to numeric, standardises them (Section 2.8 shows what happens without this step), and sets the diagnosis aside in a variable that no algorithm will see until Section 2.8. The figure plots two features as the algorithm sees them — unlabelled — and, beside them, as a supervised method would.



In [ ]:
data(BreastCancer, package = "mlbench")
bc <- BreastCancer[, -1]                                   # drop the Id column
bc <- bc[complete.cases(bc), ]                             # drop the 16 rows with missing values
bc[, 1:9] <- lapply(bc[, 1:9], function(z) as.numeric(as.character(z)))  # ordinal -> numeric
X_raw <- as.matrix(bc[, 1:9]); y_hidden <- bc$Class        # label used ONLY to check results, never to fit
X_std <- scale(X_raw)                                      # every feature to mean 0, sd 1
cat("Data matrix:", nrow(X_raw), "x", ncol(X_raw),
    "| after standardisation max|mean| =", round(max(abs(colMeans(X_std))), 12),
    ", min sd =", round(min(apply(X_std, 2, sd)), 6), "\n")
print(table(diagnosis = y_hidden))

par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
plot(jitter(X_raw[, "Cell.size"]), jitter(X_raw[, "Cl.thickness"]), col = GREY, pch = 19, cex = 0.5,
     xlab = "cell size", ylab = "clump thickness", main = "what an unsupervised method sees")
cols <- ifelse(y_hidden == "malignant", ORANGE, BLUE)
plot(jitter(X_raw[, "Cell.size"]), jitter(X_raw[, "Cl.thickness"]), col = cols, pch = 19, cex = 0.5,
     xlab = "cell size", ylab = "clump thickness", main = "the same cases with the hidden label")
legend("bottomright", c("benign", "malignant"), col = c(BLUE, ORANGE), pch = 19, bty = "n")



<p class="caption"><b>Figure 1.</b> Left: 683 aspirates described by cell size and clump thickness, as an unsupervised algorithm receives them — one grey cloud (jittered, because the scores are integers), denser at the lower left. Right: the same points coloured by the biopsy result that has been hidden. Malignant nuclei (orange) cluster at high cell size and high clump thickness, benign ones (blue) at low; the separation is clearer than in the two-feature view of the Python edition because these nine cytological scores were chosen by a pathologist precisely to separate the classes. Whether a clustering algorithm will split the cloud along the diagnosis, along some other direction, or at all, is the question of this unit.</p>

The printout records the curation: 683 complete cases (the 16 rows with a missing <em>Bare.nuclei</em> value were dropped), 444 benign and 239 malignant, every standardised column with mean 0 and standard deviation 1. Standardisation matters because every method in this unit is built on Euclidean distances or on variances, and both are dominated by whichever feature has the largest numbers. Here the nine features already share the 1–10 scale, so the effect is mild — Section 2.8 and Section 4.3 measure exactly how mild, which is itself the lesson.

## 2. k-Means

### 2.1 Definition: the within-cluster sum of squares

Given \(n\) points \(\mathbf{x}_1, \dots, \mathbf{x}_n \in \mathbb{R}^p\) and a chosen number of clusters \(k\), k-means seeks a partition of the points into \(k\) disjoint sets \(C_1, \dots, C_k\) and \(k\) centroids \(\boldsymbol{\mu}_1, \dots, \boldsymbol{\mu}_k\) that minimise the <strong>within-cluster sum of squares</strong> (WCSS; R's <code>kmeans</code> reports it as <code>tot.withinss</code>):

<div class="eq">\[ J(C, \boldsymbol{\mu}) = \sum_{j=1}^{k} \sum_{\mathbf{x}_i \in C_j} \lVert \mathbf{x}_i - \boldsymbol{\mu}_j \rVert_2^2 \]
<span class="where">where \(\lVert \mathbf{x}_i - \boldsymbol{\mu}_j \rVert_2^2\) is the squared Euclidean distance between a point and the centroid of the cluster it belongs to. For a fixed partition, \(J\) is minimised when each \(\boldsymbol{\mu}_j\) is the arithmetic mean of the points in \(C_j\) — hence the name — and for fixed centroids, \(J\) is minimised when each point is assigned to its nearest centroid.</span></div>

The objective has a second reading that explains what k-means is good at. Expanding the squares, \(J\) is proportional to the sum over clusters of the <em>total pairwise squared distance within the cluster</em> divided by the cluster size, so k-means is looking for groups of points that are mutually close, measured in squared Euclidean distance. It follows that the method prefers clusters that are round (isotropic), of similar spread and of similar size — the three preferences that Section 2.7 turns into failure modes.

Finding the partition that minimises \(J\) exactly is NP-hard, even for \(k = 2\) in general dimension, so every practical algorithm is a heuristic that finds a <em>local</em> minimum. The name "k-means" was coined by MacQueen in 1967 for an online version of the procedure;<sup class="c"><a href="#r5">[5]</a></sup> the batch algorithm that everyone uses was described by Lloyd in a 1957 Bell Laboratories report on pulse-code modulation, published only in 1982,<sup class="c"><a href="#r6">[6]</a></sup> and is known as <strong>Lloyd's algorithm</strong>. Section 2.3 states it; Section 2.4 executes it by hand.

### 2.2 Types of k-means

The textbook algorithm has several variants, each addressing one of its limitations. The lecture lists them under "types"; one paragraph each, with the situation that calls for it.

**Hard k-means (the standard).** Each point belongs to exactly one cluster. This is what R's <code>stats::kmeans</code> implements and what the rest of this section uses. Use it when the clusters are expected to be compact, roughly spherical and of comparable size, and when a crisp partition is what is needed.

**Soft or fuzzy k-means (fuzzy c-means).** Each point receives a <em>membership degree</em> \(u_{ij} \in [0, 1]\) in every cluster, with \(\sum_j u_{ij} = 1\), and the objective becomes \(\sum_i \sum_j u_{ij}^m \lVert \mathbf{x}_i - \boldsymbol{\mu}_j \rVert^2\) with a fuzziness exponent \(m > 1\) (usually 2). Centroids are membership-weighted means. Bezdek's 1981 monograph is the standard reference.<sup class="c"><a href="#r7">[7]</a></sup> Use it when points near a boundary should not be forced into one group — tissue segmentation in magnetic-resonance images, where a voxel may contain both grey and white matter, is the classical biomedical application. The <code>e1071::cmeans</code> function implements it; the Gaussian mixture model of Section 6, which gives posterior probabilities of membership, is the principled probabilistic version of the same idea.

**k-medoids (PAM).** The representative of each cluster is not the mean but a <em>medoid</em> — the actual data point whose total distance to the other members is smallest — and the objective may use any dissimilarity, not only squared Euclidean distance. Partitioning Around Medoids (PAM) is the algorithm of Kaufman and Rousseeuw,<sup class="c"><a href="#r8">[8]</a></sup> available in R as <code>cluster::pam</code>. Use it when the data are not Euclidean (sequence-alignment distances; patient dissimilarities built from mixed categorical and numerical variables), when robustness to outliers matters (a medoid cannot be dragged by one extreme point), or when the representative must be a real case that a clinician can inspect. Its cost is \(O(k(n-k)^2)\) per iteration, so it needs approximations (<code>cluster::clara</code>) for large \(n\).

**Mini-batch k-means.** Each iteration updates the centroids with a small random subset of the data instead of the whole sample, using a per-centroid learning rate that decreases with the number of points already assigned to it.<sup class="c"><a href="#r9">[9]</a></sup> Use it when \(n\) is in the millions (pixels of a whole-slide image, reads of a sequencing run) and a slightly worse objective is an acceptable price for a speed-up of one or two orders of magnitude.

**k-means++ initialisation.** Not a different objective but a different <em>start</em>: the first centroid is a random data point and each subsequent centroid is drawn with probability proportional to its squared distance from the nearest centroid already chosen, so that the initial centroids are spread out. Arthur and Vassilvitskii proved that this seeding alone gives an expected objective within a factor \(O(\log k)\) of the optimum, and in practice it removes most of the bad local minima that plain random initialisation falls into.<sup class="c"><a href="#r10">[10]</a></sup> Base R's <code>kmeans</code> does not ship it, so Section 2.5 implements it in a few lines and measures what it buys.

**Spherical k-means.** Each point is first normalised to unit length and the dissimilarity is the cosine distance \(1 - \cos\theta\) rather than the Euclidean distance, so that only the <em>direction</em> of a feature vector matters, not its magnitude. Use it when the total is arbitrary but the profile is informative: gene-expression profiles with varying sequencing depth, spectra of different overall intensity, word counts. In R, normalise each row to unit length before <code>kmeans</code>.

### 2.3 Lloyd's algorithm

<ol>
<li><strong>Initialise.</strong> Choose \(k\) initial centroids \(\boldsymbol{\mu}_1^{(0)}, \dots, \boldsymbol{\mu}_k^{(0)}\) — classically, \(k\) data points picked at random; today, by k-means++.</li>
<li><strong>Assignment step.</strong> Assign every point to the cluster of its nearest centroid: \(C_j \leftarrow \{\mathbf{x}_i : \lVert \mathbf{x}_i - \boldsymbol{\mu}_j \rVert \le \lVert \mathbf{x}_i - \boldsymbol{\mu}_l \rVert \;\forall\, l\}\).</li>
<li><strong>Update step.</strong> Recompute every centroid as the mean of its cluster: \(\boldsymbol{\mu}_j \leftarrow \frac{1}{|C_j|}\sum_{\mathbf{x}_i \in C_j} \mathbf{x}_i\).</li>
<li><strong>Repeat</strong> steps 2–3 until no assignment changes (or the centroids move less than a tolerance, or a maximum number of iterations is reached).</li>
</ol>

Each step can only decrease \(J\) or leave it unchanged — the assignment step because every point moves to a closer centroid, the update step because the mean is the point that minimises the sum of squared distances to a set — and since there are finitely many partitions, the algorithm terminates. What it terminates at is a local minimum that depends on the start, which is the subject of Section 2.5. The cost of one iteration is \(O(nkp)\): every point is compared with every centroid in every dimension.

### 2.4 Worked example by hand

Six two-dimensional points and \(k = 2\). The coordinates are integers so that every distance can be computed on paper; the squared distance is used throughout because the nearest centroid is the same whether the distance or its square is compared, and squares avoid roots.

| point | \(x_1\) | \(x_2\) |
|---|---|---|
| \(\mathbf{x}_1\) | 1 | 1 |
| \(\mathbf{x}_2\) | 2 | 1 |
| \(\mathbf{x}_3\) | 1 | 2 |
| \(\mathbf{x}_4\) | 6 | 5 |
| \(\mathbf{x}_5\) | 7 | 5 |
| \(\mathbf{x}_6\) | 6 | 6 |

**Initialisation.** Deliberately poor, to show the algorithm recover: \(\boldsymbol{\mu}_1^{(0)} = \mathbf{x}_1 = (1, 1)\) and \(\boldsymbol{\mu}_2^{(0)} = \mathbf{x}_2 = (2, 1)\), two neighbouring points of the same visual group.

**Iteration 1 — assignment.** Squared distances \(d^2(\mathbf{x}_i, \boldsymbol{\mu}_j) = (x_{i1} - \mu_{j1})^2 + (x_{i2} - \mu_{j2})^2\):

| point | \(d^2\) to \(\boldsymbol{\mu}_1 = (1, 1)\) | \(d^2\) to \(\boldsymbol{\mu}_2 = (2, 1)\) | assigned to |
|---|---|---|---|
| \((1, 1)\) | \(0\) | \(1\) | \(C_1\) |
| \((2, 1)\) | \(1\) | \(0\) | \(C_2\) |
| \((1, 2)\) | \(1\) | \(2\) | \(C_1\) |
| \((6, 5)\) | \(41\) | \(32\) | \(C_2\) |
| \((7, 5)\) | \(52\) | \(41\) | \(C_2\) |
| \((6, 6)\) | \(50\) | \(41\) | \(C_2\) |

The objective at this point is the sum of the chosen squared distances: \(J = 0 + 0 + 1 + 32 + 41 + 41 = 115\).

**Iteration 1 — update.** \(\boldsymbol{\mu}_1 = \) mean of \(\{(1,1), (1,2)\} = (1, 1.5)\); \(\boldsymbol{\mu}_2 = \) mean of \(\{(2,1), (6,5), (7,5), (6,6)\} = (5.25, 4.25)\).

**Iteration 2 — assignment.** Point \(\mathbf{x}_2 = (2,1)\) is now \(1.25\) from \(\boldsymbol{\mu}_1\) but \(21.125\) from \(\boldsymbol{\mu}_2\), so it moves to \(C_1\); the three far points stay in \(C_2\). The objective falls to \(J = 0.25 + 1.25 + 0.25 + 1.125 + 3.625 + 3.625 = 10.125\).

**Iteration 2 — update.** \(\boldsymbol{\mu}_1 = \) mean of \(\{(1,1), (2,1), (1,2)\} = (\tfrac{4}{3}, \tfrac{4}{3}) \approx (1.33, 1.33)\); \(\boldsymbol{\mu}_2 = \) mean of \(\{(6,5), (7,5), (6,6)\} = (\tfrac{19}{3}, \tfrac{16}{3}) \approx (6.33, 5.33)\).

A third assignment step would change nothing, so the algorithm has <strong>converged</strong> after two iterations. The final objective is \(J^\ast = \tfrac{24}{9} = \tfrac{8}{3} \approx 2.667\). The sequence \(115 \to 10.125 \to 2.667\) is the monotone decrease promised in Section 2.3. The chunk below implements Lloyd's algorithm in a dozen lines of base R and prints the same tables.



In [ ]:
# Lloyd's algorithm in base R, traced on the six hand-worked points with the same (poor) start.
P <- matrix(c(1,1, 2,1, 1,2, 6,5, 7,5, 6,6), ncol = 2, byrow = TRUE)
lloyd <- function(P, centroids, max_iter = 20, verbose = TRUE) {
  history <- numeric(0)
  for (it in seq_len(max_iter)) {
    d2  <- sapply(seq_len(nrow(centroids)), function(j) rowSums(sweep(P, 2, centroids[j, ])^2))
    lab <- max.col(-d2, ties.method = "first")                      # assignment step: nearest centroid
    history <- c(history, sum(d2[cbind(seq_len(nrow(P)), lab)]))    # objective J after assignment
    if (verbose) {
      tab <- data.frame(round(d2, 3)); names(tab) <- paste0("d2_mu", seq_len(ncol(d2)))
      tab$assigned <- lab; rownames(tab) <- paste0("x", seq_len(nrow(P)))
      cat(sprintf("iteration %d: J = %.3f\n", it, history[it])); print(tab)
    }
    new <- t(sapply(seq_len(nrow(centroids)), function(j) colMeans(P[lab == j, , drop = FALSE])))
    if (all(abs(new - centroids) < 1e-12)) break                    # update step; stop when stable
    centroids <- new
  }
  list(labels = lab, centroids = centroids, history = history)
}
res <- lloyd(P, P[c(1, 2), ])
cat("\nfinal centroids:", paste(round(res$centroids, 3), collapse = " "),
    "| J sequence:", paste(round(res$history, 3), collapse = " -> "), "\n")



The printout reproduces both hand tables — including the move of \(\mathbf{x}_2\) at iteration 2 — and the objective sequence \(115 \to 10.125 \to 2.667\); the loop exits on the third pass, the convergence check in which no centroid moves. The next chunk gives R's own <code>kmeans</code> the same two initial centroids and draws the trajectory.



In [ ]:
km_hand <- kmeans(P, centers = P[c(1, 2), ], algorithm = "Lloyd", iter.max = 20)
cat("kmeans() centroids:", paste(round(km_hand$centers, 3), collapse = " "),
    "| tot.withinss (J*):", round(km_hand$tot.withinss, 3), "\n")
# centroid positions after 0, 1, 2 updates, for the picture
path <- list(P[c(1, 2), ])
for (t in 1:2) {
  d2 <- sapply(1:2, function(j) rowSums(sweep(P, 2, path[[t]][j, ])^2)); lab <- max.col(-d2)
  path[[t + 1]] <- t(sapply(1:2, function(j) colMeans(P[lab == j, , drop = FALSE])))
}
par(mar = c(4, 4, 2, 1))
plot(P, pch = 19, cex = 1.6, col = PALETTE[res$labels], xlim = c(0, 8), ylim = c(0, 7),
     xlab = expression(x[1]), ylab = expression(x[2]), main = "Lloyd's algorithm: centroids after 0, 1, 2 updates")
text(P[, 1], P[, 2], paste0("x", 1:6), pos = 1, cex = 0.8)
for (j in 1:2) {
  cx <- sapply(path, function(m) m[j, 1]); cy <- sapply(path, function(m) m[j, 2])
  lines(cx, cy, lty = 2, col = c(BLUE, ORANGE)[j])
  points(cx, cy, pch = 4, lwd = 2, cex = c(1, 1.4, 2), col = c(BLUE, ORANGE)[j])
}
legend("topleft", c("cluster C1", "cluster C2"), col = c(BLUE, ORANGE), pch = 19, bty = "n")



<p class="caption"><b>Figure 2.</b> The six points coloured by their final cluster, and the two centroids (crosses, larger as the iterations advance) moving from the poor start \(\boldsymbol{\mu}_1^{(0)} = (1,1)\), \(\boldsymbol{\mu}_2^{(0)} = (2,1)\) to the final positions \((1.33, 1.33)\) and \((6.33, 5.33)\). The orange centroid travels far: after the first assignment it is pulled to the middle of the plane by the three distant points, and only the second iteration delivers it to the right group. R's <code>kmeans</code> reports the same centroids and the same objective \(8/3 \approx 2.667\) as the from-scratch implementation.</p>

### 2.5 Local minima and initialisation

Lloyd's algorithm descends to the nearest local minimum of \(J\), and \(J\) has many. The classic bad outcome is two centroids sharing one true group while another true group is split or absorbed by a neighbour; the algorithm cannot escape, because moving a centroid across the gap between groups would first <em>increase</em> \(J\). The standard remedies are (i) a better start, k-means++, and (ii) <em>restarts</em>: run the algorithm from several initialisations and keep the solution with the smallest \(J\) — R's <code>nstart</code> argument. The chunk below generates five well-separated Gaussian groups, then compares twelve single runs from plain random starts with twelve from a from-scratch k-means++ seeding.



In [ ]:
make_blobs <- function(n, centers, sd = 1.2) {           # a base-R stand-in for sklearn's make_blobs
  k <- nrow(centers); per <- rep(n %/% k, k); if (n %% k) per[seq_len(n %% k)] <- per[seq_len(n %% k)] + 1
  list(X = do.call(rbind, lapply(seq_len(k), function(j)
         cbind(rnorm(per[j], centers[j, 1], sd), rnorm(per[j], centers[j, 2], sd)))),
       y = rep(seq_len(k), per))
}
kpp_init <- function(X, k, seed) {                        # k-means++ seeding, from scratch
  set.seed(seed); cent <- X[sample(nrow(X), 1), , drop = FALSE]
  for (j in 2:k) { d2 <- apply(X, 1, function(x) min(colSums((t(cent) - x)^2)))
                   cent <- rbind(cent, X[sample(nrow(X), 1, prob = d2), , drop = FALSE]) }
  cent
}
set.seed(0)
centres5 <- matrix(c(0,0, 9,0, 0,9, 9,9, 4.5,16), ncol = 2, byrow = TRUE)
b <- make_blobs(500, centres5, 1.2); Xb <- b$X
rand_J <- sapply(1:12, function(s) { set.seed(s); kmeans(Xb, 5, nstart = 1, algorithm = "Lloyd", iter.max = 50)$tot.withinss })
kpp_J  <- sapply(1:12, function(s) { set.seed(s); kmeans(Xb, kpp_init(Xb, 5, s), nstart = 1, iter.max = 50)$tot.withinss })
best <- min(c(rand_J, kpp_J)); thr <- 1.2 * best        # "stuck" = objective more than 20% above the best found
print(rbind(`random init`    = round(rand_J, 0), `k-means++ init` = round(kpp_J, 0)))
cat(sprintf("\nbest J = %.0f; runs stuck in a clearly worse local minimum: random %d/12, k-means++ %d/12\n",
            best, sum(rand_J > thr), sum(kpp_J > thr)))

worst <- which.max(rand_J)
par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
set.seed(worst); bad <- kmeans(Xb, 5, nstart = 1, algorithm = "Lloyd", iter.max = 50)
good <- kmeans(Xb, kpp_init(Xb, 5, 1), nstart = 1, iter.max = 50)
plot(Xb, col = PALETTE[bad$cluster], pch = 19, cex = 0.5, asp = 1, xlab = "", ylab = "",
     main = sprintf("random start, J = %.0f", bad$tot.withinss)); points(bad$centers, pch = 4, lwd = 2, cex = 1.6)
plot(Xb, col = PALETTE[good$cluster], pch = 19, cex = 0.5, asp = 1, xlab = "", ylab = "",
     main = sprintf("k-means++, J = %.0f", good$tot.withinss)); points(good$centers, pch = 4, lwd = 2, cex = 1.6)



<p class="caption"><b>Figure 3.</b> Left: a bad random-start run (seed 2). Two centroids share one group and one centroid sits between two others — a local minimum from which Lloyd's algorithm cannot escape, with an objective nearly four times the best found. Right: a k-means++ run, which recovers the five groups. The table shows the spread: from plain random starts two of the twelve runs land at \(J \approx 5200\text{–}5400\), roughly \(3.7\times\) the optimum \((J \approx 1429)\), while all twelve k-means++ runs reach the optimum. k-means++ does not guarantee the global minimum on harder data, which is why it is combined with <code>nstart</code> restarts, but it converts a frequent failure into a rare one at negligible cost.</p>

<div class="box warn"><h4>Common misconception</h4><p>"k-means found the clusters" is a statement about one run from one start. The same data and the same \(k\) can produce different partitions from different starts, and the printed objective is the only internal evidence of which is better. Always run several initialisations (<code>nstart = 10</code> or more in R), always report the objective, and never interpret a partition that another start improves upon.</p></div>

### 2.6 Choosing k

The number of clusters is an input, not an output, and no internal criterion can determine it with certainty — the data do not know how many groups the analyst wants. Three heuristics are in common use.

**The elbow method.** Plot the objective \(J^\ast(k)\) against \(k\). It always decreases (with \(k = n\) it is zero), but if the data have \(k_0\) groups it drops steeply up to \(k_0\) and slowly thereafter; the "elbow" is the value where the gain per added cluster collapses. The method is visual, often ambiguous, and should be read as a range rather than a value.

**The silhouette.** For each point \(i\), let \(a(i)\) be its mean distance to the other members of its own cluster and \(b(i)\) the smallest mean distance to the members of any other cluster. The silhouette of the point is

<div class="eq">\[ s(i) = \frac{b(i) - a(i)}{\max\{a(i), b(i)\}} \in [-1, 1] \]
<span class="where">near 1 when the point is much closer to its own cluster than to the nearest other one, near 0 when it lies between two clusters, and negative when it is probably in the wrong cluster. The mean silhouette over all points is a score for the whole partition; the \(k\) that maximises it is chosen.</span></div>

Rousseeuw introduced the silhouette in 1987, and the mean silhouette has become the most used internal index.<sup class="c"><a href="#r11">[11]</a></sup> R computes it with <code>cluster::silhouette</code>; its cost is \(O(n^2)\) because it needs all pairwise distances.

**The gap statistic.** Tibshirani, Walther and Hastie proposed to compare the observed \(\log J^\ast(k)\) with its expectation under a <em>null</em> reference distribution with no cluster structure — uniform data over the bounding box of the sample — estimated from \(B\) simulated datasets:<sup class="c"><a href="#r12">[12]</a></sup>

<div class="eq">\[ \mathrm{Gap}(k) = \mathbb{E}^\ast\!\left[\log J^\ast_{\text{null}}(k)\right] - \log J^\ast(k) \]
<span class="where">The chosen \(k\) is the one that maximises the gap (or, in the stricter rule, the smallest \(k\) within one standard error of the maximum). Unlike the elbow and the silhouette, the gap statistic can return \(k = 1\) — that is, it can say "there are no clusters" — which is often the correct answer.</span></div>



In [ ]:
db <- dist(Xb); ks <- 1:9
inertia <- sapply(ks, function(k) if (k == 1) sum(scale(Xb, scale = FALSE)^2)
                  else { set.seed(0); kmeans(Xb, k, nstart = 5)$tot.withinss })
sil <- c(NA, sapply(2:9, function(k) { set.seed(0); mean(silhouette(kmeans(Xb, k, nstart = 5)$cluster, db)[, 3]) }))
gap_statistic <- function(X, ks, B = 20) {               # Gap(k) and its s.e. from B uniform reference sets
  lo <- apply(X, 2, min); hi <- apply(X, 2, max); gaps <- errs <- numeric(0)
  for (k in ks) {
    set.seed(0); logW <- log(kmeans(X, k, nstart = 3)$tot.withinss)
    logWn <- sapply(1:B, function(b) { set.seed(100 + b)
      Xn <- cbind(runif(nrow(X), lo[1], hi[1]), runif(nrow(X), lo[2], hi[2])); log(kmeans(Xn, k, nstart = 3)$tot.withinss) })
    gaps <- c(gaps, mean(logWn) - logW); errs <- c(errs, sd(logWn) * sqrt(1 + 1 / B))
  }
  list(gap = gaps, se = errs)
}
g <- gap_statistic(Xb, ks)
cat("silhouette chooses k =", which.max(sil), "| gap is maximised at k =", which.max(g$gap), "| true number of groups = 5\n")
par(mfrow = c(1, 3), mar = c(4, 4, 2, 1))
plot(ks, inertia, type = "b", pch = 19, col = BLUE, xlab = "k", ylab = "J*(k)", main = "elbow: objective"); abline(v = 5, lty = 3, col = GREY)
plot(2:9, sil[-1], type = "b", pch = 19, col = ORANGE, xlab = "k", ylab = "mean silhouette", main = "silhouette"); abline(v = 5, lty = 3, col = GREY)
plot(ks, g$gap, type = "b", pch = 19, col = GREEN, xlab = "k", ylab = "Gap(k)", main = "gap statistic (+/- s.e.)")
arrows(ks, g$gap - g$se, ks, g$gap + g$se, length = 0.02, angle = 90, code = 3, col = GREEN); abline(v = 5, lty = 3, col = GREY)



<p class="caption"><b>Figure 4.</b> Three criteria on data generated with five groups (dotted line). The objective curve has its elbow at \(k = 5\) — the decrease from 4 to 5 is large, from 5 to 6 small. The mean silhouette peaks at \(k = 5\). The gap statistic is flat and low up to \(k = 4\), because for fewer than five clusters the within-cluster scatter stays nearly as large as for uniform data, then jumps at \(k = 5\) where each group finally gets a centroid, so its maximum falls at the true value. The three agree here because the groups are well separated and round; on real data they frequently disagree, and the analyst reports the disagreement rather than the criterion that gave the desired answer. (The strict "first within one s.e." rule is fragile when the increments at small \(k\) are tiny relative to the standard error, as they are here; the gap <em>maximum</em> is unambiguous.)</p>

### 2.7 Failure modes, demonstrated

Section 2.1 noted that the objective prefers round clusters of similar size and spread. The chunk below constructs three datasets that violate one preference each and shows what k-means does with them, scored by the adjusted Rand index (Section 2.8).



In [ ]:
make_moons <- function(n, noise = 0.06) {                 # interleaved half-moons, a base-R make_moons
  n1 <- n %/% 2; n2 <- n - n1; t1 <- seq(0, pi, length.out = n1); t2 <- seq(0, pi, length.out = n2)
  rbind(cbind(cos(t1), sin(t1)), cbind(1 - cos(t2), 0.5 - sin(t2))) + matrix(rnorm(2 * n, 0, noise), ncol = 2)
}
set.seed(0)
Xa <- rbind(cbind(rnorm(600, 0, 2), rnorm(600, 0, 2)), cbind(rnorm(40, 7, .5), rnorm(40, 5, .5)), cbind(rnorm(40, 7, .5), rnorm(40, -5, .5)))
ya <- rep(1:3, c(600, 40, 40))
Xb_moons <- make_moons(400, 0.06); yb_moons <- rep(1:2, c(200, 200))
base_c <- rbind(cbind(rnorm(150, 0, 3), rnorm(150, 0, .6)), cbind(rnorm(150, 0, 3), rnorm(150, 3, .6)))  # groups differ along feature 2
yc <- rep(1:2, c(150, 150)); Xc <- base_c %*% diag(c(50, 1))                                             # feature 1: large irrelevant scale
par(mfrow = c(1, 3), mar = c(4, 4, 3, 1))
for (d in list(list("(a) unequal sizes", Xa, ya, 3), list("(b) non-spherical", Xb_moons, yb_moons, 2), list("(c) unscaled features", Xc, yc, 2))) {
  set.seed(0); km <- kmeans(d[[2]], d[[4]], nstart = 10)
  plot(d[[2]], col = PALETTE[km$cluster], pch = 19, cex = 0.5, xlab = "", ylab = "",
       main = sprintf("%s\nARI = %.2f", d[[1]], adjusted_rand(d[[3]], km$cluster))); points(km$centers, pch = 4, lwd = 2, cex = 1.6)
}
cat("(c) after standardising both features: ARI =", round(adjusted_rand(yc, kmeans(scale(Xc), 2, nstart = 10)$cluster), 2), "\n")



<p class="caption"><b>Figure 5.</b> Three failures. (a) One large, diffuse group and two small, tight ones: k-means with \(k = 3\) places a centroid on each small group but assigns to it a wide fringe of the large group as well, because the boundary between two centroids is the perpendicular bisector of the segment joining them, which falls far inside the diffuse group; the low ARI counts the misassigned points. (b) Two half-moons: k-means can only draw straight boundaries between centroids, so it cuts each moon in half (ARI near 0.2). (c) Two round groups separated along feature 2, but feature 1 is measured on a scale fifty times larger and is irrelevant to the grouping: the Euclidean distance is dominated by feature 1 and the split is made along its noise, ignoring the real separation (ARI near 0); standardising both features restores the correct partition (printed ARI \(\approx\) 0.99).</p>

Each failure has a remedy elsewhere in the unit: unequal spreads are handled by Gaussian mixtures, which fit a covariance per cluster (Section 6); non-convex shapes by single-linkage hierarchical clustering or spectral clustering (Sections 3 and 6); unscaled features by standardisation, which is the reason Section 1.3 standardised the breast-cancer data before anything else.

### 2.8 k-Means versus the hidden labels on the breast-cancer data

The running example is finally clustered. k-means with \(k = 2\) is run on the nine standardised features, and the partition is compared with the biopsy result that the algorithm never saw. The comparison uses a <strong>contingency table</strong> — cluster against label — and the <strong>adjusted Rand index</strong> (ARI) of Hubert and Arabie,<sup class="c"><a href="#r13">[13]</a></sup> which counts pairs of cases that the two partitions agree to put together or apart, corrected so that a random partition scores 0 and a perfect match scores 1. The ARI is invariant to the names of the clusters, which is essential: k-means calls its groups 1 and 2 in an arbitrary order that has nothing to do with "malignant" and "benign".



In [ ]:
set.seed(0); km2 <- kmeans(X_std, centers = 2, nstart = 10)
print(table(`k-means cluster` = km2$cluster, biopsy = y_hidden))
agree <- max(mean(km2$cluster == as.integer(y_hidden)), mean(km2$cluster != as.integer(y_hidden)))
cat(sprintf("\nadjusted Rand index = %.3f\n", adjusted_rand(y_hidden, km2$cluster)))
cat(sprintf("agreement with the biopsy under the best relabelling = %.1f%%  (NOT an accuracy: no label was used to fit)\n", 100 * agree))
set.seed(0); km2_raw <- kmeans(X_raw, centers = 2, nstart = 10)
cat(sprintf("the same with the UNstandardised features: ARI = %.3f, cluster sizes = %s\n",
            adjusted_rand(y_hidden, km2_raw$cluster), paste(as.vector(table(km2_raw$cluster)), collapse = ", ")))



On the standardised features one cluster holds 434 of the 444 benign cases and the other holds 220 of the 239 malignant ones; the adjusted Rand index is about 0.84 and, relabelling the clusters as diagnoses after the fact, about 96 % of cases fall on the "right" side. Here is the promised contrast with the Python edition: on the unstandardised features the ARI is <em>still</em> about 0.85 and the partition is only slightly different. The reason is structural — the nine mlbench features are all cytological scores on the same 1–10 scale, so no single feature can dominate the distance the way <em>worst area</em> in µm² dominated the 30-feature data. Standardisation is good practice and costs nothing, but it rescues the result only when the raw features are on incommensurable scales, which is exactly the situation of Figure 5c and <em>not</em> the situation here. The honest statement is that for this particular dataset the step barely matters, and knowing why is worth more than a reflex to always scale.

<div class="box warn"><h4>The correction from Unit 2, restated</h4><p>The 96 % agreement is not an accuracy and k-means is not a classifier. A classifier learns a mapping from features to <em>the label</em>; k-means learned a partition of the feature space that minimises within-cluster scatter and happens — because nuclear size and shape are the dominant source of variation in these data, and also the basis of the diagnosis — to align with the label. Three consequences follow. (i) The alignment cannot be known without labels, so in a genuinely unlabelled study it cannot be claimed. (ii) The partition will not improve with more cases, because no information about the label enters it. (iii) A supervised classifier on the same data reaches higher agreement (Units 3–4) because it is optimising the right thing. The correct use of this result is the one in Section 8: when the labels do <em>not</em> exist, clustering is how one discovers that the cases fall into groups worth naming.</p></div>

## 3. Hierarchical clustering

### 3.1 The agglomerative algorithm

k-means needs \(k\) in advance and returns one partition. Hierarchical clustering returns <em>all</em> partitions at once, nested inside each other, as a tree — the <strong>dendrogram</strong> — and \(k\) is chosen afterwards by cutting the tree at a height. The <strong>agglomerative</strong> (bottom-up) algorithm is:

<ol>
<li>Start with \(n\) clusters, one per point, and the \(n \times n\) matrix of pairwise distances.</li>
<li>Find the two clusters at the smallest distance and <strong>merge</strong> them. Record the merge and its distance (the <em>height</em> of the merge).</li>
<li>Update the distance matrix: compute the distance between the new cluster and every remaining cluster using a <strong>linkage</strong> rule.</li>
<li>Repeat steps 2–3 until one cluster remains. The \(n - 1\) recorded merges are the dendrogram.</li>
</ol>

Everything that distinguishes one hierarchical method from another is in step 3 — the definition of the distance between two <em>clusters</em>, given the distances between their <em>points</em>. The naive implementation costs \(O(n^3)\) in time and \(O(n^2)\) in memory for the distance matrix, which limits the method to some tens of thousands of cases; for larger \(n\) one clusters a sample or the centroids of a preliminary k-means. In R the algorithm is <code>hclust</code>, fed a <code>dist</code> object.

### 3.2 Distances between clusters: the linkage rules

For clusters \(A\) and \(B\) with \(|A|\) and \(|B|\) points and pairwise point distances \(d(\mathbf{a}, \mathbf{b})\):

| Linkage (R `method=`) | \(D(A, B)\) | Character |
|---|---|---|
| **Single** `"single"` | \(\min_{\mathbf{a} \in A,\, \mathbf{b} \in B} d(\mathbf{a}, \mathbf{b})\) | Merges clusters that touch anywhere; follows elongated shapes; prone to <em>chaining</em> (Section 3.5) |
| **Complete** `"complete"` | \(\max_{\mathbf{a} \in A,\, \mathbf{b} \in B} d(\mathbf{a}, \mathbf{b})\) | Merges only clusters whose furthest points are close; compact, round clusters; sensitive to outliers |
| **Average** `"average"` (UPGMA) | \(\dfrac{1}{|A|\,|B|} \sum_{\mathbf{a}} \sum_{\mathbf{b}} d(\mathbf{a}, \mathbf{b})\) | A compromise; the method of the phylogenetic trees of BGM324 and of the gene-expression heatmaps of Section 3.6 |
| **Ward** `"ward.D2"` | \(\sqrt{\dfrac{2\,|A|\,|B|}{|A| + |B|}}\; \lVert \bar{\mathbf{a}} - \bar{\mathbf{b}} \rVert_2\) | Merges the pair whose union increases the within-cluster sum of squares least; the hierarchical counterpart of k-means |

Merging \(A\) and \(B\) increases the within-cluster sum of squares by \(\frac{|A||B|}{|A|+|B|} \lVert \bar{\mathbf{a}} - \bar{\mathbf{b}} \rVert^2\) (with centroids \(\bar{\mathbf{a}}, \bar{\mathbf{b}}\)); R's <code>"ward.D2"</code> reports the square root of twice that quantity, so that for two single points it equals their Euclidean distance. Ward's 1963 paper defines the method as the merge that minimises the loss of an objective function, the error sum of squares being the case that became standard.<sup class="c"><a href="#r15">[15]</a></sup> All four rules are special cases of the Lance–Williams recurrence,<sup class="c"><a href="#r14">[14]</a></sup> which expresses the distance between a merged cluster \(A \cup B\) and any other cluster \(C\) as a linear combination of \(D(A, C)\), \(D(B, C)\) and \(D(A, B)\), so the distance matrix can be updated without returning to the points. (R's historical <code>"ward.D"</code> omits the square root and should not be used; <code>"ward.D2"</code> is the correct implementation of Ward's criterion.)

### 3.3 Worked example by hand: the first three merges

Eight points in two dimensions, coordinates to one decimal so every distance can be checked on paper. Three tight pairs sit far apart, with two further isolated points. The chunk prints the points and the six smallest pairwise distances; the hand trace follows.



In [ ]:
H <- matrix(c(1.0,1.0, 1.3,1.0,  5.0,5.0, 5.6,5.0,  1.0,5.0, 1.0,5.8,  3.0,3.0, 7.0,1.0),
            ncol = 2, byrow = TRUE, dimnames = list(paste0("x", 1:8), c("x1", "x2")))
Dm <- as.matrix(dist(H)); Dm[lower.tri(Dm, diag = TRUE)] <- NA
pairs <- which(!is.na(Dm), arr.ind = TRUE)
ord <- order(Dm[!is.na(Dm)]); six <- pairs[order(Dm[pairs]), ][1:6, ]
cat("the eight points:\n"); print(H)
cat("\nsix smallest pairwise distances:\n")
for (r in 1:6) cat(sprintf("  x%d - x%d : %.3f\n", six[r, 1], six[r, 2], Dm[six[r, 1], six[r, 2]]))



**Merge 1.** The smallest distance is \(d(\mathbf{x}_1, \mathbf{x}_2) = \sqrt{0.3^2 + 0^2} = 0.3\). Cluster \(\{1, 2\}\) forms at height 0.3.

**Merge 2.** The next smallest is \(d(\mathbf{x}_3, \mathbf{x}_4) = \sqrt{0.6^2} = 0.6\). Because \(\mathbf{x}_3\) and \(\mathbf{x}_4\) are isolated from everything else, no linkage rule can make the cluster \(\{1, 2\}\) closer to any point than 0.6, so this merge is \(\{3, 4\}\) at height 0.6 for <em>every</em> linkage.

**Merge 3.** By the same reasoning, \(d(\mathbf{x}_5, \mathbf{x}_6) = \sqrt{0.8^2} = 0.8\): the third merge is \(\{5, 6\}\) at height 0.8, again for every linkage.

**Where the linkages diverge.** The rules agree on these three merges only because the pairs are isolated. Consider the updated distance between cluster \(\{1, 2\}\) and point \(\mathbf{x}_7 = (3, 3)\): \(d(\mathbf{x}_1, \mathbf{x}_7) = \sqrt{2^2 + 2^2} = 2.828\) and \(d(\mathbf{x}_2, \mathbf{x}_7) = \sqrt{1.7^2 + 2^2} = 2.625\). Single linkage records \(\min = 2.625\), complete linkage \(\max = 2.828\), average linkage \((2.625 + 2.828)/2 = 2.727\), and Ward, with the centroid of \(\{1, 2\}\) at \((1.15, 1)\), records \(\sqrt{\tfrac{2 \cdot 2 \cdot 1}{3}}\, \lVert (1.15,1) - (3,3) \rVert = 1.155 \times 2.724 = 3.146\). As merges accumulate, these differences compound and the four dendrograms become different trees. The chunk confirms the three merge heights and the four updated distances.



In [ ]:
hc_methods <- c("single", "complete", "average", "ward.D2")
first3 <- sapply(hc_methods, function(m) round(hclust(dist(H), m)$height[1:3], 3))
cat("first three merge heights by linkage (rows = merge 1,2,3):\n"); print(first3)
A <- H[c(1, 2), ]; p7 <- H[7, ]; dp <- sqrt(rowSums(sweep(A, 2, p7)^2))
cat(sprintf("\nD({1,2}, x7): single = %.3f, complete = %.3f, average = %.3f, ward = %.3f\n",
            min(dp), max(dp), mean(dp), sqrt(2 * 2 * 1 / 3) * sqrt(sum((colMeans(A) - p7)^2))))
par(mfrow = c(1, 2), mar = c(2, 4, 2, 1))
plot(hclust(dist(H), "average"), main = "average linkage", xlab = "", sub = "", ylab = "merge height")
plot(hclust(dist(H), "ward.D2"), main = "Ward linkage", xlab = "", sub = "", ylab = "merge height")



<p class="caption"><b>Figure 6.</b> Dendrograms of the eight points under average linkage (left) and Ward linkage (right). Reading from the bottom, the lowest three joins are the three merges traced by hand — \(\{1,2\}\) at 0.3, \(\{3,4\}\) at 0.6, \(\{5,6\}\) at 0.8 — identical in both trees; higher up the trees differ in the order and height of the later merges, and Ward's heights grow faster because its criterion penalises merging large clusters. The confirmed \(D(\{1,2\}, \mathbf{x}_7)\) values — 2.625, 2.828, 2.727, 3.146 for single, complete, average and Ward — are exactly the hand calculation.</p>

### 3.4 Cutting the tree and the cophenetic correlation

A dendrogram becomes a partition when it is cut horizontally: every branch that crosses the cut height is one cluster (<code>cutree(hc, k)</code> cuts to obtain \(k\) clusters). The "right" height is often read from the dendrogram as the longest vertical stretch with no merges — a large gap between successive merge heights means that the clusters below it are much more similar internally than to each other.

A dendrogram also implies a distance between any two points: the height at which they are first joined, the <strong>cophenetic distance</strong>. The <strong>cophenetic correlation coefficient</strong> is the Pearson correlation between the cophenetic distances and the original distances over all pairs;<sup class="c"><a href="#r16">[16]</a></sup> it measures how faithfully the tree represents the distance matrix and is the standard way to compare linkage rules on the same data (values above about 0.75 indicate a faithful representation). R provides <code>cophenetic(hc)</code>. The chunk draws a 20-case subset of the breast-cancer data on two features, cuts each tree into two clusters, compares with the hidden labels, and reports the cophenetic correlation of each linkage.



In [ ]:
set.seed(0); si <- sample(nrow(X_raw), 20)
sub <- X_raw[si, c("Cell.size", "Cl.thickness")]; sub_lab <- y_hidden[si]; dsub <- dist(sub)
tab <- data.frame(t(sapply(hc_methods, function(m) {
  hc <- hclust(dsub, m); cut <- cutree(hc, 2)
  c(coph = round(cor(cophenetic(hc), dsub), 3),
    ARI  = round(adjusted_rand(sub_lab, cut), 3),
    size1 = min(table(cut)), size2 = max(table(cut)),
    max_gap = round(max(diff(hc$height)), 3))
})))
print(tab)



The table makes three points. First, the cophenetic correlation is highest for single or average linkage here (all four are close, 0.85–0.87, because the 20-point subset is small and its distances modest); average linkage is, by construction, the rule that best preserves the original distances in general, which is why it is the default of phylogenetics and of the heatmaps of Section 3.6. Second, cutting the trees at \(k = 2\) gives partitions of different balance and agreement with the diagnosis — single linkage in particular tends to peel off one or two outlying cases and leave the rest together, the small-scale version of the chaining effect shown next. Third, the "max_gap" column is a crude automatic reading of the dendrogram; it is a heuristic, not a test.

### 3.5 The chaining effect of single linkage

Single linkage merges two clusters as soon as <em>any</em> pair of their points is close. A thin line of points between two groups — a few intermediate patients, a noisy transition, a bridge — is therefore enough to join the groups, link by link, before the groups themselves are complete. The resulting clusters are long chains rather than compact sets. The same property is a virtue when the clusters genuinely are elongated or curved (the half-moons of Figure 5b), so the effect is a trade-off, not a bug.



In [ ]:
set.seed(0)
g1 <- cbind(rnorm(100, 0, .5), rnorm(100, 0, .5)); g2 <- cbind(rnorm(100, 6, .5), rnorm(100, 0, .5))
bridge <- cbind(seq(0.8, 5.2, length.out = 12), rnorm(12, 0, .05))
Xc2 <- rbind(g1, g2, bridge)
par(mfrow = c(1, 3), mar = c(2, 2, 3, 1))
for (m in c("single", "average", "ward.D2")) {
  lab <- cutree(hclust(dist(Xc2), m), 2)
  plot(Xc2, col = PALETTE[lab], pch = 19, cex = 0.5, asp = 1, xlab = "", ylab = "",
       main = sprintf("%s, k = 2\nsizes %s", m, paste(as.vector(table(lab)), collapse = ", ")))
  cat(sprintf("%-8s on the half-moons (k = 2): ARI = %.2f\n", m,
              adjusted_rand(yb_moons, cutree(hclust(dist(Xb_moons), m), 2))))
}



<p class="caption"><b>Figure 7.</b> Two hundred points in two tight groups, plus twelve points forming a bridge. Single linkage (left) follows the bridge and joins both groups into one cluster, leaving a single outlier as the "second cluster". Average and Ward linkage (centre, right) cut the bridge and recover the two groups. The printout shows the other side of the trade-off: on the half-moons of Figure 5b, single linkage is the only one of the three that recovers the two curved shapes (ARI \(\approx 1\)), because chaining is exactly what following a curve requires, while average and Ward score around 0.4 and 0.5.</p>

**Divisive methods.** The opposite strategy — start with one cluster containing everything and split recursively — is <strong>divisive</strong> hierarchical clustering. It is rarely used in its exact form because choosing the best split of a cluster is combinatorial (there are \(2^{n-1} - 1\) ways to split \(n\) points in two); the practical algorithm DIANA<sup class="c"><a href="#r8">[8]</a></sup> (available as <code>cluster::diana</code>) splits by moving the most dissimilar points out one at a time, and <em>bisecting k-means</em> splits the largest or least compact cluster with \(k = 2\) at each step. Divisive methods get the top of the tree — the coarse structure — right first, whereas agglomerative methods, having committed to early merges, cannot undo them.

### 3.6 Application: gene-expression heatmaps

In 1998 Eisen, Spellman, Brown and Botstein applied average-linkage hierarchical clustering with a correlation-based distance to the expression of thousands of yeast genes across time courses, reordered the rows and columns of the expression matrix by the dendrograms, and displayed the result as a colour image — the "clustered heatmap" that has been the standard visualisation of expression data ever since.<sup class="c"><a href="#r17">[17]</a></sup> The idea is simple: genes with similar expression profiles end up adjacent, so blocks of colour appear where groups of genes behave alike in groups of conditions, and both the gene clusters and the sample clusters can be read off the margins. The Python edition built its miniature on scikit-learn's wine chemistry table<sup class="c"><a href="#r18">[18]</a></sup> as a stand-in for a samples × genes matrix; R has no such dataset built in, so the chunk <em>simulates</em> a small expression-style matrix with three groups of samples and correlated blocks of features — which is in fact closer to the Eisen application than the wine data were — and clusters both axes with average linkage on the correlation distance, exactly as Eisen did. R's <code>heatmap</code> performs the clustering and the reordering in one call.



In [ ]:
set.seed(0)
# simulated expression matrix: 60 samples in 3 groups x 18 features in 3 co-varying blocks
make_expr <- function() {
  grp <- rep(1:3, each = 20); block_means <- matrix(c(2,-1,-1, -1,2,-1, -1,-1,2), 3, 3, byrow = TRUE)
  M <- matrix(0, 60, 18)
  for (g in 1:3) for (bk in 1:3) {
    cols <- ((bk - 1) * 6 + 1):(bk * 6)
    M[grp == g, cols] <- rnorm(20 * 6, block_means[g, bk], 0.7)
  }
  list(M = scale(M), grp = grp)
}
ex <- make_expr(); colnames(ex$M) <- paste0("g", 1:18); rownames(ex$M) <- paste0("s", 1:60)
cc_dist <- function(x) as.dist(1 - cor(t(x)))                        # correlation distance between rows
side <- c(BLUE, ORANGE, GREEN)[ex$grp]
heatmap(ex$M, distfun = cc_dist, hclustfun = function(d) hclust(d, "average"),
        RowSideColors = side, col = colorRampPalette(c(BLUE, "white", ORANGE))(40),
        scale = "none", margins = c(4, 4), xlab = "features (genes)", ylab = "samples")
cut3 <- cutree(hclust(cc_dist(ex$M), "average"), 3)
cat("three sample clusters vs. the hidden group: ARI =", round(adjusted_rand(ex$grp, cut3), 2),
    "; cluster sizes", paste(as.vector(table(cut3)), collapse = ", "), "\n")



<p class="caption"><b>Figure 8.</b> A clustered heatmap in the style of Eisen et al. Each row is a simulated sample, each column a standardised feature (orange above the mean, blue below); rows and columns have been reordered by average-linkage dendrograms on the correlation distance, drawn on the margins. Blocks of colour appear, one per group of samples sharing a high block of features, because the simulation built exactly that structure. The coloured bar at the left shows the hidden group of each sample and reveals that the sample clusters coincide with the three groups (ARI printed above). What the figure does <em>not</em> show is the raw matrix in its original order, which would be noise to the eye; the structure is produced by the reordering.</p>

<div class="box bridge"><h4>Bridge to bioinformatics</h4><p>The UPGMA tree of BGM324 Unit 1, built from a matrix of pairwise sequence distances, <em>is</em> average-linkage agglomerative clustering: the leaves are sequences, the merge heights are evolutionary distances, and the cophenetic distances are the branch lengths. The clustered heatmap is the same algorithm applied twice, to the rows and to the columns of an expression matrix, and Section 8 describes how it defined the molecular subtypes of breast cancer.</p></div>

## 4. Principal component analysis

### 4.1 Two views of the same idea

Principal component analysis (PCA) replaces \(p\) correlated features by \(q \le p\) new features — the <strong>principal components</strong> — that are linear combinations of the originals, uncorrelated with each other, and ordered by the amount of variance they carry. Two formulations lead to the same components.

**Variance maximisation.** Find the unit vector \(\mathbf{w}_1\) such that the projection of the (centred) data onto it, \(z_i = \mathbf{w}_1^\top \mathbf{x}_i\), has the largest possible variance; then find \(\mathbf{w}_2 \perp \mathbf{w}_1\) with the largest remaining variance, and so on. This is Hotelling's 1933 formulation,<sup class="c"><a href="#r19">[19]</a></sup> and the one from which the method takes its name.

**Minimum reconstruction error.** Find the \(q\)-dimensional subspace (a line for \(q = 1\), a plane for \(q = 2\)) such that the sum of squared perpendicular distances from the points to the subspace is smallest. This is Pearson's 1901 formulation — "lines and planes of closest fit to systems of points in space"<sup class="c"><a href="#r20">[20]</a></sup> — and it says that PCA is the best possible <em>linear compression</em> of the data: the \(q\) coordinates of each point in that subspace are the summary from which the point can be reconstructed with the least total squared error.

The two agree because, for centred data, the squared distance from a point to the origin equals the squared length of its projection on the subspace plus the squared perpendicular distance to it; the first sum is fixed, so maximising the projected variance and minimising the residual are the same problem. Both lead to the covariance matrix.

### 4.2 Covariance, eigenvectors, eigenvalues — worked example by hand

Let \(\mathbf{X}\) be the \(n \times p\) data matrix with column means subtracted. Its <strong>sample covariance matrix</strong> is

<div class="eq">\[ \mathbf{S} = \frac{1}{n-1}\,\mathbf{X}^\top \mathbf{X}, \qquad S_{jl} = \frac{1}{n-1}\sum_{i=1}^{n} x_{ij}\,x_{il} \]
<span class="where">a symmetric \(p \times p\) matrix whose diagonal holds the variances of the features and whose off-diagonal entries hold their covariances. The variance of a projection \(\mathbf{w}^\top \mathbf{x}\) is \(\mathbf{w}^\top \mathbf{S}\, \mathbf{w}\), and maximising it subject to \(\lVert \mathbf{w} \rVert = 1\) (by a Lagrange multiplier) gives \(\mathbf{S}\mathbf{w} = \lambda \mathbf{w}\): the directions of maximal variance are the <strong>eigenvectors</strong> of \(\mathbf{S}\), and the variance along each is its <strong>eigenvalue</strong> \(\lambda\). Because \(\mathbf{S}\) is symmetric, its eigenvectors are orthogonal and its eigenvalues real and non-negative. Sorted in decreasing order, \(\lambda_1 \ge \dots \ge \lambda_p\), the eigenvectors are the principal components, and \(\lambda_j / \sum_l \lambda_l\) is the <strong>proportion of variance explained</strong> by component \(j\).</span></div>

**Worked example.** Five points in two dimensions: \((2,1), (3,3), (4,2), (5,4), (6,5)\).

**Step 1 — centre.** Means \(\bar{x}_1 = 4\), \(\bar{x}_2 = 3\). Centred data: \((-2, -2), (-1, 0), (0, -1), (1, 1), (2, 2)\).

**Step 2 — covariance matrix.** With \(n - 1 = 4\): \(S_{11} = (4 + 1 + 0 + 1 + 4)/4 = 2.5\); \(S_{22} = 2.5\); \(S_{12} = (4 + 0 + 0 + 1 + 4)/4 = 2.25\). So \(\mathbf{S} = \begin{pmatrix} 2.5 & 2.25 \\ 2.25 & 2.5 \end{pmatrix}\).

**Step 3 — eigenvalues.** \(\det(\mathbf{S} - \lambda \mathbf{I}) = (2.5 - \lambda)^2 - 2.25^2 = 0\), i.e. \(2.5 - \lambda = \pm 2.25\): \(\lambda_1 = 4.75\), \(\lambda_2 = 0.25\). The trace is preserved: \(4.75 + 0.25 = 5 = S_{11} + S_{22}\). Proportion explained by PC1: \(4.75 / 5 = 0.95\).

**Step 4 — eigenvectors.** For \(\lambda_1 = 4.75\): \(-2.25 w_1 + 2.25 w_2 = 0\), so \(\mathbf{w}_1 = (1, 1)/\sqrt{2} \approx (0.707, 0.707)\). For \(\lambda_2 = 0.25\): \(\mathbf{w}_2 = (1, -1)/\sqrt{2} \approx (0.707, -0.707)\), orthogonal to \(\mathbf{w}_1\).

**Step 5 — project.** The PC1 score of each centred point is \(z_{i1} = (x_{i1} + x_{i2})/\sqrt{2} \approx -2.83, -0.71, -0.71, 1.41, 2.83\); their variance is \(4.75 = \lambda_1\). The PC2 scores are \(0, -0.71, 0.71, 0, 0\), whose squares sum to \(1.0\) — the total squared reconstruction error if the data are compressed to one dimension. The chunk confirms every number.



In [ ]:
Q  <- matrix(c(2,1, 3,3, 4,2, 5,4, 6,5), ncol = 2, byrow = TRUE)
Qc <- scale(Q, center = TRUE, scale = FALSE)         # step 1: centre
S  <- t(Qc) %*% Qc / (nrow(Q) - 1)                   # step 2: covariance matrix
eg <- eigen(S)                                        # steps 3-4: eigen-decomposition (descending)
scores <- Qc %*% eg$vectors                           # step 5: project
cat("covariance matrix:\n"); print(S)
cat("eigenvalues:", round(eg$values, 4), "| proportion explained:", round(eg$values / sum(eg$values), 3), "\n")
cat("eigenvectors (columns):\n"); print(round(eg$vectors, 4))
cat("PC1 scores:", round(scores[, 1], 3), "| var:", round(var(scores[, 1]), 3), "\n")
cat("sum of squared PC2 scores (1-D reconstruction error):", round(sum(scores[, 2]^2), 3), "\n")
pr <- prcomp(Q)
cat("prcomp eigenvalues (sdev^2):", round(pr$sdev^2, 4), "| rotation PC1:", round(pr$rotation[, 1], 4), "\n")



<code>eigen</code> and <code>prcomp</code> reproduce the hand calculation: eigenvalues 4.75 and 0.25, the proportions 0.95 and 0.05, eigenvectors \(\pm(0.707, 0.707)\) and \(\pm(0.707, -0.707)\), and the scores. The signs of the eigenvectors may differ between the hand solution, <code>eigen</code> and <code>prcomp</code> — an eigenvector multiplied by \(-1\) is still an eigenvector — so a component's sign carries no meaning and the scores along it may be flipped from one run or one function to another. This is the first of the two sign/scale ambiguities (the second belongs to ICA, Section 5.3).



In [ ]:
w1 <- eg$vectors[, 1]; proj <- sweep(outer(scores[, 1], w1), 2, colMeans(Q), "+")  # reconstruct from PC1 only
par(mar = c(4, 4, 2, 1))
plot(Q, pch = 19, cex = 1.4, asp = 1, xlim = c(0.5, 7.5), ylim = c(-0.5, 6.5),
     xlab = expression(x[1]), ylab = expression(x[2]), main = "PCA of five points: axis of maximal variance = line of closest fit")
m <- colMeans(Q)
for (sgn in c(-1, 1)) { segments(m[1], m[2], m[1] + sgn * 2.6 * w1[1], m[2] + sgn * 2.6 * w1[2], col = BLUE, lwd = 2)
                        segments(m[1], m[2], m[1] + sgn * 1.2 * eg$vectors[1,2], m[2] + sgn * 1.2 * eg$vectors[2,2], col = ORANGE, lwd = 2) }
for (i in 1:5) segments(Q[i, 1], Q[i, 2], proj[i, 1], proj[i, 2], col = GREY, lty = 3)
points(proj, pch = 19, col = BLUE, cex = 0.9); points(m[1], m[2], pch = 3, cex = 2, lwd = 2)
legend("topleft", c("data", "reconstruction from PC1", "PC1 (95%)", "PC2 (5%)"),
       col = c("black", BLUE, BLUE, ORANGE), pch = c(19, 19, NA, NA), lty = c(NA, NA, 1, 1), lwd = 2, bty = "n", cex = 0.8)



<p class="caption"><b>Figure 9.</b> The five points (black), their mean (cross), the first principal axis (blue, along \((1,1)/\sqrt{2}\)) and the second (orange, perpendicular). The dotted segments are the perpendicular distances from each point to the first axis; their squared lengths sum to 1.0, the reconstruction error of Step 5. The blue dots on the axis are the points as they would be reconstructed from a single number each, their PC1 score. Hotelling's view (maximal variance along the blue axis) and Pearson's view (minimal squared dotted distances) describe the same line.</p>

### 4.3 Explained variance and the scree plot: with and without scaling

With nine features the covariance matrix is \(9 \times 9\) and has nine eigenvalues. The <strong>scree plot</strong> draws them (or the proportions of variance) in decreasing order; the number of components to keep is read either from an elbow in the plot or from the cumulative proportion crossing a threshold such as 80 % or 95 %. The chunk computes the eigenvalues for the raw features and for the standardised ones.



In [ ]:
pca_raw <- prcomp(X_raw, center = TRUE, scale. = FALSE)
pca_std <- prcomp(X_raw, center = TRUE, scale. = TRUE)
vr_raw <- pca_raw$sdev^2 / sum(pca_raw$sdev^2); vr_std <- pca_std$sdev^2 / sum(pca_std$sdev^2)
cat(sprintf("RAW:          PC1 %.1f%%, PC2 %.1f%%; PC1 dominated by '%s'\n",
            100 * vr_raw[1], 100 * vr_raw[2], names(which.max(abs(pca_raw$rotation[, 1])))))
cat(sprintf("STANDARDISED: PC1 %.1f%%, PC2 %.1f%%, PC3 %.1f%%; first two %.1f%%; %d components reach 95%%\n",
            100 * vr_std[1], 100 * vr_std[2], 100 * vr_std[3], 100 * sum(vr_std[1:2]), which(cumsum(vr_std) >= 0.95)[1]))
par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
for (z in list(list(vr_raw, "raw features"), list(vr_std, "standardised features"))) {
  barplot(z[[1]], names.arg = 1:9, col = BLUE, ylim = c(0, 1), xlab = "principal component", ylab = "proportion of variance", main = z[[2]])
  lines(seq(0.7, by = 1.2, length.out = 9), cumsum(z[[1]]), type = "b", pch = 19, col = ORANGE); abline(h = 0.95, lty = 3, col = GREY)
}



<p class="caption"><b>Figure 10.</b> Scree plots. Because the nine mlbench features already share the 1–10 scale, the raw and standardised pictures are similar: PC1 carries about 69 % of the variance raw and 66 % standardised, and seven components reach 95 % either way. This is the opposite of the Python edition's 30-feature result, where the raw PC1 reached 98 % and was almost entirely <em>worst area</em> in µm². Here the raw PC1 is merely dominated by <em>Bare.nuclei</em>, the feature with the largest raw variance, and standardising removes even that mild imbalance. The single large first component — two thirds of the variance — reflects a real biological fact: the nine cytological scores are strongly positively correlated, because a malignant nucleus tends to be abnormal on all of them at once.</p>

<div class="box key"><h4>Standardise before PCA</h4><p>PCA maximises variance, and variance has units. A feature measured in µm² has a variance in µm⁴ and will dominate features measured as ratios whatever their biological importance. Unless all features share a unit and their variances are meaningfully comparable — as, unusually, they are here — standardise first, which is the same as performing PCA on the <em>correlation</em> matrix (<code>prcomp(..., scale. = TRUE)</code>) instead of the covariance matrix. Jolliffe and Cadima's review discusses this and the other practical decisions of the method.<sup class="c"><a href="#r21">[21]</a></sup></p></div>

### 4.4 Loadings and the biplot

The entries of each eigenvector are the <strong>loadings</strong>: the weight of each original feature in the component. Reading them is how a component is interpreted. The <strong>biplot</strong> superimposes the scores of the cases on the first two components (points) with the loadings of the features (arrows): a long arrow means the feature contributes strongly; arrows pointing the same way mean correlated features; the position of a case along an arrow indicates a high or low value of that feature.



In [ ]:
load <- pca_std$rotation[, 1:2]
cat("|loading| on PC1 (sorted):\n"); print(round(sort(abs(load[, 1]), decreasing = TRUE), 3))
cat("largest |loading| on PC2:\n"); print(round(sort(abs(load[, 2]), decreasing = TRUE)[1:3], 3))
sc <- pca_std$x[, 1:2]
par(mar = c(4, 4, 2, 1))
plot(sc, col = GREY, pch = 19, cex = 0.4, xlab = sprintf("PC1 (%.0f%%)", 100 * vr_std[1]),
     ylab = sprintf("PC2 (%.0f%%)", 100 * vr_std[2]), main = "Biplot of the standardised breast-cancer data")
abline(h = 0, v = 0, col = GREY, lwd = 0.5); scl <- 4.5
arrows(0, 0, scl * load[, 1], scl * load[, 2], col = ORANGE, lwd = 1.4, length = 0.08)
text(scl * load[, 1] * 1.14, scl * load[, 2] * 1.14, seq_len(nrow(load)), col = ORANGE, font = 2, cex = 0.9)  # numbered tips
legend("topright", legend = paste0(seq_len(nrow(load)), ". ", rownames(load)), text.col = ORANGE, bty = "n", cex = 0.72, title = "feature loadings")



<p class="caption"><b>Figure 11.</b> Biplot. The 683 cases (grey) are plotted by their scores on PC1 and PC2; the orange arrows are the loadings of the nine features (numbered, with the key at the right), scaled for visibility. Eight of the nine arrows point in almost the same direction along PC1: <em>cell size</em>, <em>cell shape</em>, <em>bland chromatin</em>, <em>epithelial cell size</em>, <em>normal nucleoli</em>, <em>bare nuclei</em>, <em>marginal adhesion</em> and <em>clump thickness</em> all load on PC1 with similar weights (0.30–0.38), so PC1 is essentially the <em>average</em> of the cytological scores — a general "nuclear abnormality" axis, and the direction along which benign cases (left) separate from malignant ones (right). The ninth feature, <em>Mitoses</em>, points almost straight up: it dominates PC2 (loading 0.91) because it is the score least correlated with the rest, high only in a minority of aggressive tumours. No label was used, yet the hidden diagnosis is largely a function of PC1 — which is why k-means found it in Section 2.8.</p>

The loadings table confirms the reading: the eight size-and-texture scores load on PC1 with nearly equal weight, so PC1 is close to their mean, while PC2 is almost entirely <em>Mitoses</em>. An interpretation of this kind is the main scientific use of PCA: it tells the analyst that the nine features measure, to a first approximation, two things — a dominant degree-of-abnormality and a largely independent mitotic signal.

### 4.5 PCA as a singular value decomposition; whitening

R's <code>prcomp</code> does not form the covariance matrix. It computes the <strong>singular value decomposition</strong> (SVD) of the centred data matrix, \(\mathbf{X} = \mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^\top\), where \(\mathbf{V}\) holds the right singular vectors (the principal components, as columns), \(\boldsymbol{\Sigma}\) the singular values \(\sigma_j\), and \(\mathbf{U}\boldsymbol{\Sigma}\) the scores. The connection is \(\mathbf{X}^\top \mathbf{X} = \mathbf{V} \boldsymbol{\Sigma}^2 \mathbf{V}^\top\), so the eigenvalues of \(\mathbf{S}\) are \(\lambda_j = \sigma_j^2 / (n - 1)\). The SVD route is numerically more stable and avoids the \(p \times p\) matrix, which matters when \(p\) is in the thousands (genes).

<strong>Whitening</strong> (or sphering) goes one step further: divide each score by its standard deviation \(\sqrt{\lambda_j}\), so that the transformed data have identity covariance — every direction unit variance, all uncorrelated. Whitening throws away the variance information that PCA ranks by, which makes it useless for choosing components but essential as the first step of ICA (Section 5).



In [ ]:
Xc <- scale(X_std, center = TRUE, scale = FALSE); sv <- svd(Xc)
cat("eigenvalues from SVD, sigma^2/(n-1):", round(sv$d[1:4]^2 / (nrow(Xc) - 1), 3), "\n")
cat("eigenvalues from prcomp:            ", round(prcomp(X_std)$sdev[1:4]^2, 3), "\n")
pr <- prcomp(X_std); scr <- pr$x[, 1:5]; whi <- sweep(scr, 2, pr$sdev[1:5], "/")
cat("covariance of the first five scores (diagonal = eigenvalues, off-diagonal ~ 0):\n"); print(round(cov(scr), 2))
cat("covariance of the five whitened scores (identity):\n"); print(round(cov(whi), 2))



The singular values squared and divided by \(n - 1\) are the eigenvalues to every printed digit. The covariance of the scores is diagonal — the components are uncorrelated by construction, with the eigenvalues on the diagonal — and after whitening it is the identity.

### 4.6 PCA for visualisation and as preprocessing

Two practical uses follow. First, <strong>visualisation</strong>: projecting high-dimensional data onto the first two components gives the two-dimensional picture that retains the most variance. R has no handwritten-digits dataset (the Python edition used scikit-learn's <code>load_digits</code><sup class="c"><a href="#r22">[22]</a></sup>), so the chunk <em>simulates</em> six classes of 8 × 8 images — a vertical bar, a horizontal bar, a diagonal, a filled block, a frame and an anti-diagonal, each with pixel noise — and projects them, exactly as the Python edition projected the real digits.



In [ ]:
set.seed(0)
templ <- list(function(){m<-matrix(0,8,8); m[,4:5]<-1; c(m)}, function(){m<-matrix(0,8,8); m[4:5,]<-1; c(m)},
              function(){m<-matrix(0,8,8); diag(m)<-1; c(m)}, function(){m<-matrix(0,8,8); m[3:6,3:6]<-1; c(m)},
              function(){m<-matrix(0,8,8); m[1,]<-m[8,]<-m[,1]<-m[,8]<-1; c(m)}, function(){m<-matrix(0,8,8); m[cbind(1:8,8:1)]<-1; c(m)})
dig <- do.call(rbind, lapply(templ, function(f) t(replicate(60, f() + rnorm(64, 0, 0.15))))); dlab <- rep(1:6, each = 60)
pd <- prcomp(dig); Zd <- pd$x[, 1:2]
cat("variance retained by two components:", round(sum((pd$sdev^2 / sum(pd$sdev^2))[1:2]), 3),
    "| components for 90%:", which(cumsum(pd$sdev^2 / sum(pd$sdev^2)) >= 0.9)[1], "of 64\n")
par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
plot(Zd, col = PALETTE[dlab], pch = 19, cex = 0.5, xlab = "PC1", ylab = "PC2", main = "simulated images in the PC1-PC2 plane")
for (d in 1:6) text(mean(Zd[dlab==d, 1]), mean(Zd[dlab==d, 2]), d, font = 2, cex = 1.3)
image(1:8, 1:8, matrix(pd$rotation[, 1], 8, 8)[, 8:1], col = colorRampPalette(c(BLUE, "white", ORANGE))(30),
      xlab = "", ylab = "", main = "PC1 as an 8x8 image", axes = FALSE)



<p class="caption"><b>Figure 12.</b> Left: the 360 simulated images in the plane of their first two components, coloured and numbered by the pattern class (hidden from PCA). Two components retain just over half the variance, yet the six classes already form separated regions. Right: the first component displayed as an 8 × 8 image — a component of an image dataset is itself an image, and reading it this way is the pixel-space equivalent of reading a loadings table. The patterns are simulated; the pedagogy (PCA for visualisation, components as images) is identical to the Python edition's handwritten digits.</p>

Second, <strong>preprocessing</strong>: replacing the \(p\) features by the first \(q\) components before a supervised classifier removes redundancy, reduces noise and speeds up distance-based methods such as k-NN. The chunk measures the 5-fold cross-validated accuracy of 5-NN on the breast-cancer data as a function of the number of components retained — with the PCA fitted <em>inside</em> each fold, because fitting it on all the data first would let the test folds influence the components (the leakage of Supplement S1, in a form often overlooked because PCA uses no labels).



In [ ]:
suppressPackageStartupMessages(library(class))
set.seed(0); folds <- sample(rep(1:5, length.out = nrow(X_raw)))
cv_knn_pca <- function(q) mean(sapply(1:5, function(f) {
  tr <- folds != f; ctr <- colMeans(X_raw[tr, ]); sdt <- apply(X_raw[tr, ], 2, sd)
  Xtr <- scale(X_raw[tr, ], ctr, sdt); Xte <- scale(X_raw[!tr, ], ctr, sdt)
  if (q < 9) { p <- prcomp(Xtr); Xtr <- Xtr %*% p$rotation[, 1:q, drop = FALSE]; Xte <- Xte %*% p$rotation[, 1:q, drop = FALSE] }
  mean(knn(Xtr, Xte, y_hidden[tr], k = 5) == y_hidden[!tr])
}))
qs <- c(1, 2, 3, 4, 5, 7, 9); accs <- sapply(qs, cv_knn_pca)
print(data.frame(components = qs, CV_accuracy = round(accs, 4)))
par(mar = c(4, 4, 2, 1)); plot(qs, accs, type = "b", pch = 19, col = BLUE, ylim = c(0.95, 0.98),
     xlab = "number of principal components q", ylab = "5-fold CV accuracy", main = "k-NN accuracy after reducing to q components")
abline(h = accs[length(accs)], col = ORANGE, lty = 2)



<p class="caption"><b>Figure 13.</b> Cross-validated accuracy of 5-NN trained on the first \(q\) principal components of the standardised features. On these data a <em>single</em> component — the general-abnormality axis of Figure 11 — already gives about 97 %, matching or slightly exceeding the full nine features (dashed line), and the curve is essentially flat thereafter. Eight of the nine dimensions can be discarded at no cost, because the diagnostic signal lives almost entirely on PC1; the nine cytological scores are, for this task, effectively one-dimensional. (The Python edition's 30-feature data needed about five components to plateau — a reminder that the number of useful dimensions is a property of the dataset, not a universal.)</p>

Note the use of the labels here. PCA itself used none — the components are the same whether or not \(\mathbf{y}\) exists — but the <em>evaluation</em> of how many components to keep for a supervised task necessarily uses the labels, through the cross-validated accuracy. That is legitimate as long as the PCA is refitted inside each fold, as the chunk does.

### 4.7 Limits of PCA, and t-SNE and UMAP for visualisation

PCA is <strong>linear</strong>: the components are weighted sums of the features, and the subspace is flat. Data on a curved surface — a spiral, a sphere, the half-moons — are poorly summarised by any flat subspace. Kernel PCA and the autoencoders of Unit 6 are the nonlinear extensions.

More important, and more often forgotten: <strong>variance is not information</strong>. PCA ranks directions by how much the data spread along them, not by how relevant that spread is to any question. If the difference between two groups lies along a direction of small variance while a large, irrelevant source (age, batch, instrument) dominates, the first components will describe the irrelevant source and the signal will be in a late component or discarded. The chunk builds exactly this situation.



In [ ]:
set.seed(0); n <- 300
irrelevant <- rnorm(n, 0, 4)                                  # e.g. patient age: large spread, unrelated to the groups
group <- sample(0:1, n, replace = TRUE)
marker <- ifelse(group == 0, -1, 1) + rnorm(n, 0, 0.4)        # informative feature: small spread, separates the groups
Xvi <- cbind(irrelevant, marker); pv <- prcomp(Xvi); Zvi <- pv$x
cat("proportion of variance:", round(pv$sdev^2 / sum(pv$sdev^2), 3), "| PC1 loadings:", round(pv$rotation[, 1], 3), "\n")
cv_glm <- function(F) { set.seed(0); fo <- sample(rep(1:5, length.out = n)); mean(sapply(1:5, function(f) {
  tr <- fo != f; m <- glm(group[tr] ~ ., data.frame(F[tr, , drop = FALSE]), family = binomial)
  mean((predict(m, data.frame(F[!tr, , drop = FALSE]), type = "response") > 0.5) == group[!tr]) })) }
cat(sprintf("CV accuracy of logistic regression:  PC1 only = %.3f | PC2 only = %.3f | both features = %.3f\n",
            cv_glm(Zvi[, 1, drop = FALSE]), cv_glm(Zvi[, 2, drop = FALSE]), cv_glm(Xvi)))
par(mfrow = c(1, 2), mar = c(4, 4, 2, 1)); gc <- ifelse(group == 0, BLUE, ORANGE)
plot(Xvi, col = gc, pch = 19, cex = 0.5, xlab = "irrelevant feature (variance 16)", ylab = "marker (variance ~1.2)", main = "original features")
plot(Zvi, col = gc, pch = 19, cex = 0.5, xlab = "PC1 (93%)", ylab = "PC2 (7%)", main = "principal components")



<p class="caption"><b>Figure 14.</b> Left: two groups (colours hidden from PCA) separated cleanly along a feature of small variance, while an unrelated feature has about fourteen times the variance. Right: PCA puts about 93 % of the variance into PC1, which is essentially the irrelevant feature; the group separation survives only in PC2. An analyst who kept "the component that explains 93 % of the variance" would have discarded the only informative direction — a logistic regression on PC1 alone performs near chance (about 57 %) while one on PC2 alone is nearly perfect, as the printout shows.</p>

For <strong>visualisation</strong> of high-dimensional data in two dimensions, two nonlinear methods have largely displaced PCA in the biomedical literature: <strong>t-SNE</strong>,<sup class="c"><a href="#r23">[23]</a></sup> which places points so that neighbours in the original space remain neighbours, and <strong>UMAP</strong>,<sup class="c"><a href="#r24">[24]</a></sup> which does the same faster and with a stronger claim to preserve coarse structure. Both produce the striking "islands" plots of single-cell RNA sequencing (R packages <code>Rtsne</code> and <code>uwot</code>). Both are <em>for visualisation only</em>: the distances between islands and their sizes are not meaningful, the result depends on tuning parameters and the random seed, and new points cannot be placed without recomputing. They should never be used as preprocessing for clustering or classification, and clusters seen in them should be confirmed in the original feature space.

<div class="box warn"><h4>Common misconception</h4><p>"The first two components explain only 30 % of the variance, so the PCA failed." A low proportion in two components means the data are genuinely high-dimensional, which is a finding, not a failure; and, as Figure 14 shows, a high proportion in one component can be a failure, when that component is irrelevant. The proportion of variance measures compression, not relevance.</p></div>

## 5. Independent component analysis

### 5.1 The cocktail-party problem; independence versus uncorrelatedness

Two people speak at once in a room with two microphones. Each microphone records a different weighted sum of the two voices, the weights depending on the distances. Given only the two recordings, can the two voices be recovered? This is the <strong>cocktail-party problem</strong>, the prototype of <strong>blind source separation</strong>: \(m\) observed signals \(\mathbf{x}(t)\) are modelled as an unknown linear mixture of \(m\) unknown source signals \(\mathbf{s}(t)\),

<div class="eq">\[ \mathbf{x} = \mathbf{A}\,\mathbf{s}, \qquad \hat{\mathbf{s}} = \mathbf{W}\,\mathbf{x} \]
<span class="where">where \(\mathbf{A}\) is the \(m \times m\) <em>mixing matrix</em>, and the task is to find an <em>unmixing matrix</em> \(\mathbf{W} \approx \mathbf{A}^{-1}\) from the observations alone, with no knowledge of \(\mathbf{A}\) or of the sources except one assumption: the sources are <strong>statistically independent</strong>.</span></div>

Independent component analysis (ICA) is the family of methods that solve this problem. The name and formal definition are due to Comon (1994);<sup class="c"><a href="#r25">[25]</a></sup> the algorithm that made it practical, Infomax, to Bell and Sejnowski (1995);<sup class="c"><a href="#r26">[26]</a></sup> and the fast fixed-point algorithm implemented below, FastICA, to Hyvärinen and Oja, whose 2000 review is the standard introduction.<sup class="c"><a href="#r27">[27]</a></sup>

The distinction that makes ICA different from PCA is between <em>uncorrelated</em> and <em>independent</em>. Two variables are uncorrelated when their covariance is zero — a statement about second moments only. They are independent when their joint distribution factorises, \(p(s_1, s_2) = p(s_1)\,p(s_2)\) — a statement about all moments. Independence implies uncorrelatedness; the converse holds only for Gaussian variables. PCA finds uncorrelated components, and for a Gaussian signal that is all there is to find; for non-Gaussian signals there is a stronger structure, independence, and PCA does not see it.

### 5.2 Non-Gaussianity as the lever

Why can independence be recovered at all? The argument is the central limit theorem run backwards. A sum of independent random variables is <em>more Gaussian</em> than any of its terms. Each observed mixture \(x_j = \sum_l a_{jl} s_l\) is such a sum, so the mixtures are closer to Gaussian than the sources. Conversely, among all linear combinations \(\mathbf{w}^\top \mathbf{x}\) of the observations, the <em>least</em> Gaussian ones must be the original sources (up to scale). ICA therefore searches for the directions \(\mathbf{w}\) that maximise a measure of non-Gaussianity, and it works exactly to the extent that the sources are non-Gaussian — <strong>at most one source may be Gaussian</strong>, or the problem has no unique solution.

Two measures of non-Gaussianity are in use. <strong>Kurtosis</strong>, the standardised fourth moment \(\kappa = \mathbb{E}[s^4]/\mathbb{E}[s^2]^2 - 3\), is zero for a Gaussian, positive for "peaky" heavy-tailed distributions (speech, spikes) and negative for flat ones (a square wave, a uniform variable); it is simple but sensitive to outliers. <strong>Negentropy</strong>, the entropy deficit relative to a Gaussian of equal variance, is zero for a Gaussian and positive otherwise, is robust, and is what FastICA approximates with a smooth contrast function such as \(\log\cosh\) (whose derivative is \(\tanh\)). FastICA first <em>whitens</em> the data with PCA (Section 4.5), so that the remaining problem is to find a <em>rotation</em>, then finds the rotation one direction at a time by a fixed-point iteration on the chosen contrast.

### 5.3 FastICA from scratch on three synthetic sources: PCA fails, ICA succeeds

There is no FastICA in base R, so the chunk implements it from scratch in about twenty lines — centre, whiten via <code>eigen</code> of the covariance, then a fixed-point iteration with \(g = \tanh\) and Gram–Schmidt deflation — exactly as the Python edition wrote its own. Three sources — a sine, a square wave and a sawtooth — are mixed by a random \(3 \times 3\) matrix into three observed channels. PCA and FastICA are each asked for three components, and the recovered components are compared with the true sources by their absolute correlation.



In [ ]:
fastica <- function(X, n_comp, max_iter = 200, tol = 1e-6, seed = 1) {
  Xc <- scale(X, center = TRUE, scale = FALSE)               # 1. centre
  e  <- eigen(cov(Xc)); K <- diag(1 / sqrt(e$values)) %*% t(e$vectors)   # 2. whitening matrix (PCA)
  Xw <- (Xc %*% t(K))[, 1:n_comp, drop = FALSE]              #    whitened data: identity covariance
  W  <- matrix(0, n_comp, n_comp); set.seed(seed)
  for (i in 1:n_comp) {                                       # 3. one independent direction at a time
    w <- rnorm(n_comp); w <- w / sqrt(sum(w^2))
    for (it in 1:max_iter) {
      wx <- as.vector(Xw %*% w); g <- tanh(wx); gp <- 1 - g^2
      wn <- colMeans(g * Xw) - mean(gp) * w                   #    fixed-point update (negentropy, g = tanh)
      if (i > 1) { Wp <- W[1:(i - 1), , drop = FALSE]; wn <- wn - t(Wp) %*% (Wp %*% wn) }  # deflation
      wn <- as.vector(wn); wn <- wn / sqrt(sum(wn^2))
      if (abs(abs(sum(wn * w)) - 1) < tol) { w <- wn; break }
      w <- wn
    }
    W[i, ] <- w
  }
  list(S = Xw %*% t(W), U = W %*% K[1:n_comp, , drop = FALSE], mu = colMeans(X))   # S = sources; U = full unmixing
}
set.seed(0); tt <- seq(0, 8, length.out = 2000)
saw <- function(x) 2 * (x / (2 * pi) - floor(0.5 + x / (2 * pi)))
S_true <- cbind(sin(2 * pi * tt), sign(sin(2 * pi * 0.6 * tt + 1)), saw(2 * pi * 0.4 * tt))
S_true <- scale(S_true + 0.05 * matrix(rnorm(length(S_true)), ncol = 3))
A_mix  <- matrix(rnorm(9), 3, 3); X_mix <- S_true %*% t(A_mix)
kurt <- function(M) apply(M, 2, function(z) mean(((z - mean(z)) / sd(z))^4) - 3)
cat("kurtosis of the sources :", round(kurt(S_true), 2), " (far from 0: non-Gaussian)\n")
cat("kurtosis of the mixtures:", round(kurt(X_mix), 2), " (closer to 0: mixing Gaussianises)\n")
S_pca <- prcomp(X_mix)$x; S_ica <- fastica(X_mix, 3)$S
mt <- function(R, nm) { C <- abs(cor(S_true, R)); dimnames(C) <- list(c("sine","square","saw"), paste0(nm, 1:3)); round(C, 3) }
cat("\nPCA |correlation| with each source:\n"); print(mt(S_pca, "PC"))
cat("\nICA |correlation| with each source:\n"); print(mt(S_ica, "IC"))



The kurtosis printout shows the central-limit argument at work: every source has a kurtosis well below zero (a sine, a square wave and a sawtooth are all flat-topped), and every mixture is closer to zero. The correlation tables show the result. Each PCA component is a blend of the three sources, with correlations spread across the row (best per source only 0.75–0.89) — PCA has found uncorrelated directions, which is not enough. Each ICA component correlates almost perfectly (above 0.98) with exactly one source and negligibly with the other two: the sources have been recovered without any knowledge of the mixing matrix.



In [ ]:
panels <- list(list("true sources s(t)", S_true), list("observed mixtures x(t)", X_mix),
               list("PCA components", S_pca), list("ICA components", S_ica))
par(mfcol = c(3, 4), mar = c(2, 2, 2, 1))
for (col in seq_along(panels)) for (row in 1:3) {
  plot(tt, panels[[col]][[2]][, row], type = "l", col = PALETTE[col], lwd = 0.7, xaxt = "n", yaxt = "n", xlab = "", ylab = "")
  if (row == 1) title(panels[[col]][[1]], cex.main = 0.95)
}



<p class="caption"><b>Figure 15.</b> Columns: the three sources; the three recorded mixtures, each a weighted sum of all sources; the three principal components, still mixtures (a sine with a square wave superposed, and so on); and the three independent components, which are the sources again — but in a different order and, for some, upside down. Compare the row order of the first and last columns.</p>

**The two ambiguities.** ICA cannot recover the <em>order</em> of the sources — any permutation of the rows of \(\mathbf{s}\) with the matching permutation of the columns of \(\mathbf{A}\) gives the same \(\mathbf{x}\) — nor their <em>scale and sign</em>, since multiplying a source by \(c\) and its column of \(\mathbf{A}\) by \(1/c\) changes nothing. FastICA fixes the scale by returning unit-variance components and leaves order and sign arbitrary. Consequently the "first independent component" means nothing (unlike the first principal component); components are identified by their content — waveform, spatial pattern, correlation with a reference — and that identification is where the analyst's knowledge enters.

### 5.4 Biomedical application: removing an artefact from a multichannel recording

Electrophysiological recordings — the EEG with its 19 to 256 scalp electrodes, the multi-lead ECG, the magnetoencephalogram — are the natural home of ICA, because each electrode records a linear mixture, through the conducting tissues, of several sources: the neural or cardiac activity of interest, and artefacts such as eye blinks, muscle activity, the heartbeat seen by the EEG, electrode drift and mains interference. Makeig and colleagues first applied ICA to EEG in 1996,<sup class="c"><a href="#r28">[28]</a></sup> and Jung and colleagues showed in 2000 that artefact components could be identified and removed — the recording reconstructed from the remaining components — more selectively than by rejecting contaminated epochs or by regression on a reference channel.<sup class="c"><a href="#r29">[29]</a></sup> This <em>ICA-based artefact rejection</em> is today a standard step in EEG preprocessing.

The chunk simulates the situation in miniature: a clean "ECG" source, an eye-blink-like artefact (three slow Gaussian bumps) and a slow baseline drift are mixed into four channels with different weights; FastICA is run with three components; the artefact component is identified by its correlation with the known blink (in practice: by its waveform and its spatial pattern), set to zero, and the channels are reconstructed. Reconstruction needs the Moore–Penrose pseudo-inverse of the unmixing matrix, <code>MASS::ginv</code>.<sup class="c"><a href="#r37">[37]</a></sup> The removal is scored by the root-mean-square deviation of each cleaned channel from its blink-free version.



In [ ]:
set.seed(0); fs <- 250; tt2 <- seq(0, 10, by = 1 / fs); beats <- seq(0.4, 10, by = 0.8)   # 75 bpm
ecg   <- rowSums(sapply(beats, function(b) exp(-((tt2-b)/.012)^2) - .3*exp(-((tt2-b-.05)/.02)^2) + .25*exp(-((tt2-b+.2)/.06)^2)))
blink <- rowSums(sapply(c(1.5, 4.2, 7.8), function(b) exp(-((tt2-b)/.12)^2)))              # three slow deflections
drift <- 0.3 * sin(2 * pi * 0.2 * tt2)                                                     # respiration-like wander
src <- cbind(ecg, blink, drift)
Mmix <- matrix(c(1,.2,.3, .8,1.2,.2, .6,.6,.5, .3,1.5,.4), 4, 3, byrow = TRUE)             # 4 channels x 3 sources
channels <- src %*% t(Mmix) + 0.03 * matrix(rnorm(length(tt2) * 4), ncol = 4)
clean_target <- channels - outer(blink, Mmix[, 2])                                         # channels without the blink
ic <- fastica(channels, 3, max_iter = 1000, seed = 2)
corr <- abs(cor(src, ic$S)); dimnames(corr) <- list(c("ecg","blink","drift"), paste0("IC", 1:3))
k_art <- which.max(corr["blink", ])                                                        # identify the blink component
S_clean <- ic$S; S_clean[, k_art] <- 0
cleaned <- sweep(S_clean %*% t(MASS::ginv(ic$U)), 2, ic$mu, "+")                            # zero it, reconstruct
rmse <- function(a, b) sqrt(colMeans((a - b)^2))
cat("|corr| of components with (ecg, blink, drift):\n"); print(round(corr, 3))
cat(sprintf("\nartefact component identified: IC%d\n", k_art))
print(data.frame(blink_weight = Mmix[, 2], RMSE_before = round(rmse(channels, clean_target), 3),
                 RMSE_after = round(rmse(cleaned, clean_target), 3), row.names = paste0("ch", 1:4)))


In [ ]:
layout(matrix(c(1,5, 2,6, 3,7, 4,8), 4, 2, byrow = TRUE), widths = c(2.2, 1))
par(mar = c(2, 3, 1.5, 1))
for (j in 1:4) {
  plot(tt2, channels[, j], type = "l", col = GREY, lwd = 0.6, xlab = "", ylab = paste0("ch", j), yaxt = "n")
  lines(tt2, cleaned[, j], col = BLUE, lwd = 0.6)
  for (b in c(1.5, 4.2, 7.8)) rect(b - .3, par("usr")[3], b + .3, par("usr")[4], col = rgb(.82,.41,.12,.12), border = NA)
  if (j == 1) legend("topright", c("recorded", "after removing artefact IC"), col = c(GREY, BLUE), lwd = 1.5, bty = "n", cex = 0.8)
}
nm <- c("ecg", "blink", "drift")
for (j in 1:3) { plot(tt2, ic$S[, j], type = "l", lwd = 0.6, col = if (j == k_art) ORANGE else GREEN, xlab = "", ylab = paste0("IC", j), yaxt = "n")
                 title(sprintf("IC%d ~ %s", j, nm[which.max(corr[, j])]), cex.main = 0.85) }
plot.new()



<p class="caption"><b>Figure 16.</b> Left: the four recorded channels (grey) and the same channels after the artefact component was zeroed and the recording reconstructed (blue). The three shaded intervals mark the blinks; in the recorded channels they appear as slow deflections of varying size — large in channels 2 and 4, which carry the largest blink weight — and in the cleaned channels they are largely gone, while the beats are untouched. Right: the three independent components, one of which (orange) is the blink; the others are the ECG and the drift. The table reports the root-mean-square deviation of each channel from its blink-free version, which falls by a factor of two to three; the residual is the price of imperfect separation in noise and of the sign/scale indeterminacy absorbed into the reconstruction.</p>

<div class="box key"><h4>Why ICA and not filtering?</h4><p>A blink and a heartbeat overlap in frequency, so no frequency filter removes one without distorting the other. ICA separates them by a different property — their statistical independence and their different <em>spatial</em> signatures across channels — which is why it needs several channels and a filter needs one. The same logic applies to the removal of the cardiac artefact from EEG, of muscle artefact, and of the maternal ECG from an abdominal fetal ECG recording.</p></div>

### 5.5 PCA versus ICA

| | PCA | ICA |
|---|---|---|
| Criterion | Maximal variance / minimal reconstruction error | Maximal statistical independence (non-Gaussianity) |
| Statistics used | Second order only (covariance) | Higher order (kurtosis, negentropy) |
| Components are | Uncorrelated and <em>orthogonal</em> | Independent; not orthogonal in general |
| Ordering | By variance; the first component is special | None; components identified by content |
| Sign / scale | Sign arbitrary; scale \(= \sqrt{\lambda}\) | Sign and scale arbitrary (unit variance by convention) |
| Solution | Unique, closed form (eigen / SVD) | Iterative; depends on start and contrast; at most one Gaussian source |
| Typical use | Compression, visualisation, noise reduction, preprocessing | Blind source separation: EEG/ECG/MEG artefacts, fMRI networks, hyperspectral unmixing |
| Assumes | Linearity; variance = relevance | Linearity; independent, non-Gaussian sources; as many channels as sources |

## 6. Other methods, briefly

**Gaussian mixture models and the EM algorithm.** Model the data as a weighted sum of \(k\) Gaussian densities, \(p(\mathbf{x}) = \sum_j \pi_j\, \mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}_j, \boldsymbol{\Sigma}_j)\), and fit the weights, means and covariances by maximum likelihood. The fitting algorithm is <strong>expectation–maximisation</strong> (EM),<sup class="c"><a href="#r30">[30]</a></sup> which alternates an E-step — compute, for every point, the posterior probability (<em>responsibility</em>) of each component — and an M-step — re-estimate each component as responsibility-weighted statistics. This is <em>soft k-means</em> exactly: k-means is the limit of EM for Gaussians with equal spherical covariances when the responsibilities are forced to 0 or 1. Because each component has its own covariance, mixtures handle clusters of different size, spread and orientation (the failure of Figure 5a), and the number of components can be chosen by a likelihood criterion (BIC). In R the package <code>mclust</code> provides it.

**DBSCAN.** Density-based spatial clustering of applications with noise<sup class="c"><a href="#r31">[31]</a></sup> defines a cluster as a maximal set of points connected through regions of high density: a point is a <em>core</em> point if at least <code>minPts</code> points lie within distance <code>eps</code> of it; core points within <code>eps</code> of each other are in the same cluster; non-core points within reach of a core point join it; everything else is labelled <em>noise</em>. It needs no \(k\), finds clusters of any shape, and is the only method here that can refuse to assign a point. Its weaknesses are the sensitivity to <code>eps</code> and its single density scale. The R package <code>dbscan</code> provides it.

**Spectral clustering.** Build a similarity graph on the points (k-nearest-neighbour or Gaussian-kernel weights), compute the first few eigenvectors of the graph Laplacian, and run k-means on the rows of the eigenvector matrix. The eigenvectors embed the data in a space where connected regions of the graph become compact groups, so k-means can then find clusters of arbitrary shape; the method is a relaxation of the graph-cut problem. It is accurate on moderate \(n\) and expensive beyond, because of the eigen-decomposition.

Neither <code>mclust</code> nor <code>dbscan</code> is preinstalled in this environment, so — as the Python edition would have, had scikit-learn lacked them — the chunk demonstrates the <em>connectivity</em> idea with the two methods that base R does provide: single-linkage hierarchical clustering (Section 3.5) and a from-scratch <strong>spectral clustering</strong> in a dozen lines. k-means stands in for the centroid-based methods. All three are run on the half-moons.



In [ ]:
spectral <- function(X, k, nn = 10) {                     # normalised-Laplacian spectral clustering, from scratch
  D <- as.matrix(dist(X)); n <- nrow(X); sigma <- median(D)
  W <- exp(-D^2 / (2 * sigma^2))
  for (i in 1:n) { th <- sort(D[i, ])[nn + 1]; W[i, D[i, ] > th] <- 0 }   # keep nn nearest neighbours
  W <- pmax(W, t(W)); dg <- rowSums(W)
  L <- diag(n) - diag(1 / sqrt(dg)) %*% W %*% diag(1 / sqrt(dg))          # normalised Laplacian
  U <- eigen(L, symmetric = TRUE)$vectors[, (n - k + 1):n]               # k smallest-eigenvalue vectors
  U <- U / sqrt(rowSums(U^2) + 1e-9); kmeans(U, k, nstart = 10)$cluster
}
set.seed(0); X_mo <- make_moons(300, 0.06); y_mo <- rep(1:2, c(150, 150))
methods <- list("k-means (centroid)" = kmeans(X_mo, 2, nstart = 10)$cluster,
                "single linkage (connectivity)" = cutree(hclust(dist(X_mo), "single"), 2),
                "spectral, 10-NN graph" = spectral(X_mo, 2))
par(mfrow = c(1, 3), mar = c(2, 2, 3, 1))
for (nm in names(methods)) { lab <- methods[[nm]]
  plot(X_mo, col = PALETTE[lab], pch = 19, cex = 0.5, asp = 1, xlab = "", ylab = "",
       main = sprintf("%s\nARI = %.2f", nm, adjusted_rand(y_mo, lab))) }



<p class="caption"><b>Figure 17.</b> The two half-moons under three methods. k-means fails (ARI about 0.2): it can only separate with a straight line, and cuts each crescent in half. Single-linkage clustering and the from-scratch spectral clustering both recover the two moons (ARI near 1), because both define clusters through <em>connectivity</em> — chains of nearby points — rather than through distance to a centre. A Gaussian mixture would improve on k-means but still fail (it can only separate with a quadratic curve); DBSCAN would succeed, for the same connectivity reason as the two shown. The lesson is the one of Section 2.7 again: the method encodes an assumption about cluster shape, and the assumption must fit the data.</p>

## 7. Evaluating unsupervised results

Section 1.2 stated the problem: there is no truth to compare with. Three kinds of evidence substitute for it.

**Internal indices** measure the geometry of the partition from the data alone. The <strong>silhouette</strong> (Section 2.6) rewards partitions whose points are closer to their own cluster than to the nearest other. The <strong>Davies–Bouldin index</strong><sup class="c"><a href="#r32">[32]</a></sup> is the average, over clusters, of the worst ratio of within-cluster scatter to between-centroid distance, \(\frac{1}{k}\sum_j \max_{l \ne j} \frac{\sigma_j + \sigma_l}{\lVert \boldsymbol{\mu}_j - \boldsymbol{\mu}_l \rVert}\); lower is better and 0 is ideal. All such indices prefer compact, round, well-separated clusters — which means they prefer the clusters that k-means produces, and will rate a correct connectivity-based partition of the half-moons poorly. An internal index measures agreement with a geometric ideal, not correctness.

**External indices** compare the partition with a reference labelling when one exists. The <strong>adjusted Rand index</strong> (Section 2.8) counts agreeing pairs with chance correction;<sup class="c"><a href="#r13">[13]</a></sup> the <strong>normalised mutual information</strong> (NMI) measures how much knowing the cluster tells you about the label, normalised to \([0, 1]\), and is preferred when the numbers of clusters and classes differ. Both are invariant to relabelling.

**Stability.** A structure that is real should reappear when the sample is perturbed. Re-run the clustering on random subsets and measure the agreement (ARI) between runs; a partition that changes substantially from one half of the data to the other describes the sample, not the population. Stability is necessary, not sufficient: k-means on uniform data is quite stable, and quite meaningless. The chunk computes the three internal and external indices for \(k = 2, \dots, 4\) on the standardised breast-cancer data (silhouette via <code>cluster::silhouette</code><sup class="c"><a href="#r38">[38]</a></sup>), then measures stability by subsampling.



In [ ]:
davies_bouldin <- function(X, lab) { k <- max(lab); cen <- t(sapply(1:k, function(j) colMeans(X[lab == j, , drop = FALSE])))
  s <- sapply(1:k, function(j) mean(sqrt(rowSums(sweep(X[lab == j, , drop = FALSE], 2, cen[j, ])^2))))
  mean(sapply(1:k, function(j) max(sapply(setdiff(1:k, j), function(l) (s[j] + s[l]) / sqrt(sum((cen[j, ] - cen[l, ])^2)))))) }
nmi <- function(a, b) { t <- table(a, b) / length(a); pa <- rowSums(t); pb <- colSums(t)
  H <- function(p) -sum(p[p > 0] * log(p[p > 0])); I <- sum(ifelse(t > 0, t * log(t / outer(pa, pb)), 0)); I / sqrt(H(pa) * H(pb)) }
d_std <- dist(X_std)
idx <- t(sapply(2:6, function(k) { set.seed(0); lab <- kmeans(X_std, k, nstart = 10)$cluster
  c(silhouette = mean(silhouette(lab, d_std)[, 3]), Davies_Bouldin = davies_bouldin(X_std, lab),
    ARI = adjusted_rand(y_hidden, lab), NMI = nmi(y_hidden, lab)) }))
rownames(idx) <- paste0("k=", 2:6); print(round(idx, 3))
set.seed(0)
for (k in c(2, 3, 4)) {
  aris <- sapply(1:20, function(r) {
    i1 <- sample(nrow(X_std), nrow(X_std) / 2); i2 <- sample(nrow(X_std), nrow(X_std) / 2); common <- intersect(i1, i2)
    k1 <- kmeans(X_std[i1, ], k, nstart = 5); k2 <- kmeans(X_std[i2, ], k, nstart = 5)
    p1 <- max.col(-sapply(1:k, function(j) rowSums(sweep(X_std[common, ], 2, k1$centers[j, ])^2)))
    p2 <- max.col(-sapply(1:k, function(j) rowSums(sweep(X_std[common, ], 2, k2$centers[j, ])^2)))
    adjusted_rand(p1, p2) })
  cat(sprintf("stability k=%d: mean ARI between two random halves = %.3f (worst %.3f)\n", k, mean(aris), min(aris)))
}



The indices agree here, which is not the normal case but is instructive. The silhouette and Davies–Bouldin both prefer \(k = 2\); the external indices (ARI, NMI) also peak at \(k = 2\), where the clusters align best with the biopsy, and fall as \(k\) grows because the extra clusters split the diagnosis groups. The stability analysis shows the two-cluster solution is highly reproducible across random halves (mean ARI about 0.98) and that stability decreases with \(k\). On this dataset all three kinds of evidence agree on two clusters — but that agreement is a fact about these 683 cases and these nine features, not a theorem.

### 7.1 The trap: validating clusters with the features that built them

A common error in the applied literature deserves its own demonstration. Having clustered patients on a set of variables, the analyst tests whether the clusters differ in those variables, finds \(p < 0.001\) everywhere, and reports that the clusters are "statistically distinct". They are — by construction. k-means <em>chose</em> the partition that maximises the separation of the groups on exactly those variables, so a test of that separation is circular, and it will be significant even when the data contain no structure at all. The chunk applies k-means to pure noise and runs the tests.



In [ ]:
set.seed(0)
X_noise <- matrix(rnorm(200 * 10), 200, 10)                 # 200 'patients', 10 'biomarkers', no structure
lab_noise <- kmeans(X_noise, 2, nstart = 10)$cluster
pvals <- sapply(1:10, function(j) t.test(X_noise[lab_noise == 1, j], X_noise[lab_noise == 2, j])$p.value)
cat("cluster sizes:", as.vector(table(lab_noise)), "\n")
cat("p-values of the 10 between-cluster t-tests on the clustering features:\n", round(pvals, 4), "\n")
cat(sprintf("features 'significant' at p < 0.05: %d of 10 -- in data with no groups\n", sum(pvals < 0.05)))
X_held <- matrix(rnorm(200 * 10), 200, 10)                  # the honest test: variables NOT used for clustering
p_held <- sapply(1:10, function(j) t.test(X_held[lab_noise == 1, j], X_held[lab_noise == 2, j])$p.value)
cat(sprintf("the same tests on 10 held-out variables: %d of 10 significant (expected about 0.5 by chance)\n", sum(p_held < 0.05)))



Half of the ten "biomarkers" differ significantly between two clusters of pure noise, at least one with a vanishingly small \(p\)-value. The remedy is in the last line: validate clusters on information that did not enter their construction — held-out variables, a later clinical outcome, an independent cohort, a response to treatment. The sepsis phenotypes of Section 8 are a model of this: derived from clinical variables at admission, they were validated by their association with mortality and treatment response, neither of which was used to build them.

<div class="box bridge"><h4>Bridge to Supplement S1</h4><p>The circularity here is the unsupervised twin of the leakage of Supplement S1. There, the test set was contaminated by information from training; here, the "test" of the clusters is contaminated by the clustering. The cure is the same: keep the evidence used to <em>build</em> a result apart from the evidence used to <em>judge</em> it.</p></div>

## 8. Medical applications

**Patient subtyping: sepsis phenotypes.** Sepsis is a syndrome, not a disease, and trials of targeted therapies have repeatedly failed on heterogeneous populations. Seymour and colleagues applied consensus k-means to 29 routinely available clinical and laboratory variables measured within six hours of hospital presentation in 20,189 patients, and identified four phenotypes — α, β, γ and δ — with different organ-dysfunction patterns, biomarker profiles and 28-day mortality (from about 2 % in α to about 32 % in δ); the phenotypes were reproduced in a second cohort of 43,086 patients and, in simulations of three completed trials, the apparent treatment effect changed with the phenotype mix.<sup class="c"><a href="#r33">[33]</a></sup> This is the use case of Section 1.2 in its purest form: no label existed, clustering proposed one, and external evidence — outcomes and treatment response, not the clustering variables — validated it.

**Molecular subtypes of tumours.** The clustered heatmap of Section 3.6 defined a field. Perou and colleagues applied hierarchical clustering to the expression of some 8,000 genes in 65 breast-tumour samples and found that the tumours fell into groups — later named luminal A, luminal B, HER2-enriched and basal-like — that differed in their expression patterns and, as later work showed, in prognosis and response to therapy;<sup class="c"><a href="#r34">[34]</a></sup> these "intrinsic subtypes" are now part of clinical practice. A year earlier, Golub and colleagues had shown on acute leukaemias that unsupervised clustering of expression profiles rediscovered the distinction between acute myeloid and acute lymphoblastic leukaemia without being told it (<em>class discovery</em>), and that a supervised classifier could then assign new patients to the classes (<em>class prediction</em>) — the two halves of the pipeline this unit and the previous ones describe.<sup class="c"><a href="#r35">[35]</a></sup>

**Population structure with PCA.** Genotype data — hundreds of thousands of single-nucleotide polymorphisms per person — are the largest routine application of PCA in biomedicine. Novembre and colleagues computed the first two principal components of the genotypes of 3,000 Europeans and found that the resulting scatter plot reproduced the map of Europe: the two components were, to a close approximation, latitude and longitude of the individuals' ancestry.<sup class="c"><a href="#r36">[36]</a></sup> The practical consequence is that the leading principal components are included as covariates in every genome-wide association study to prevent ancestry from confounding the association between genotype and disease — PCA as a tool for removing structure rather than for finding it.

<div class="box key"><h4>Reading an unsupervised-learning paper</h4><p>Workshop 3 asks you to answer a questionnaire on unsupervised methods; the questions to ask of any paper that uses them are the ones this unit has rehearsed. Which features were clustered, and were they standardised? Which method and which distance, and what shape of cluster does that method assume? How was \(k\) chosen, and how sensitive is the result to it? How stable is the partition under resampling? And, above all, what evidence <em>external to the clustering variables</em> shows that the clusters mean something? A paper that validates its clusters only with the variables that built them has demonstrated Section 7.1, not a discovery.</p></div>

## 9. Summary

You should now be able to explain, without notes:

- What unsupervised learning is, the three tasks it comprises (clustering, dimensionality reduction, density estimation), and why its results cannot be validated the way a classifier's can.
- The k-means objective — the within-cluster sum of squares — and why minimising it favours round clusters of similar size and spread.
- Lloyd's algorithm (initialise, assign, update, repeat), why each step decreases the objective, why it reaches a local minimum, and what k-means++ and restarts do about it.
- The variants: fuzzy c-means (membership degrees), k-medoids (a real case as representative; any dissimilarity; robust), mini-batch (scale), k-means++ (seeding), spherical (cosine distance) — and when each is called for.
- How to choose \(k\) with the elbow, the silhouette and the gap statistic, and why the gap statistic alone can answer "one cluster".
- The three failure modes of k-means (unequal sizes, non-convex shapes, unscaled features) and the remedy for each — and why, on the commensurable nine-feature breast-cancer data, the scaling remedy barely changes the result.
- Why the 96 % agreement between k-means clusters and the biopsy is not an accuracy, and what clustering is for when labels do not exist.
- The agglomerative algorithm, the four linkage rules and their formulas, the chaining effect of single linkage, how to cut a dendrogram, and what the cophenetic correlation measures.
- The clustered heatmap of Eisen et al. and how it is built (average linkage, correlation distance, both axes).
- PCA as variance maximisation and as minimum reconstruction error; how to compute components from the covariance matrix by hand; the scree plot, loadings and biplot; why standardisation comes first; PCA as SVD; whitening.
- Why variance is not information, and what t-SNE and UMAP are (and are not) for.
- ICA as blind source separation: independence versus uncorrelatedness, non-Gaussianity as the lever, the permutation and scale ambiguities, and artefact removal by zeroing a component.
- Gaussian mixtures as soft k-means fitted by EM; DBSCAN and spectral clustering as connectivity-based methods.
- Internal and external indices, stability, and the circularity of validating clusters with the variables that built them.

## 10. Self-assessment

<details><summary>1. A colleague reports that k-means on 25 laboratory variables "achieved 88 % accuracy" in separating diabetic from non-diabetic patients. State two things that are wrong with the sentence.</summary>k-means does not use labels, so it has no accuracy — the 88 % is agreement with a label after relabelling the clusters, and reflects that the diabetes-related variation happens to dominate those 25 variables. And the clusters are named 1 and 2 by the algorithm; the mapping to "diabetic" was chosen to maximise the agreement, which inflates it (with two clusters, the agreement cannot be below 50 %).</details>

<details><summary>2. Execute by hand one iteration of Lloyd's algorithm on the points \((0, 0), (0, 2), (5, 0), (5, 2)\) with initial centroids \((0, 0)\) and \((0, 2)\). What happens, and what does it illustrate?</summary>Assignment: \((0,0) \to \mu_1\); \((0,2) \to \mu_2\); \((5,0)\): \(d^2 = 25\) vs \(29 \to \mu_1\); \((5,2)\): \(29\) vs \(25 \to \mu_2\). Update: \(\mu_1 = (2.5, 0)\), \(\mu_2 = (2.5, 2)\). The next assignment changes nothing, so it converges with \(J = 4 \times 6.25 = 25\) — but the partition \(\{(0,0),(0,2)\}, \{(5,0),(5,2)\}\) has \(J = 4\). A poor start led to a local minimum; k-means++ would almost surely have chosen two far-apart initial points.</details>

<details><summary>3. Why does the elbow method always produce a decreasing curve, and why can the gap statistic return \(k = 1\) while the silhouette cannot?</summary>Adding a centroid can never increase the minimum within-cluster sum of squares (the previous solution is still available), so \(J^\ast(k)\) is non-increasing and reaches 0 at \(k = n\). The silhouette needs at least two clusters to define \(b(i)\), so it cannot evaluate \(k = 1\); the gap statistic compares \(\log J^\ast(k)\) with its value on structureless reference data and can find that no \(k > 1\) improves on \(k = 1\) beyond what uniform noise would.</details>

<details><summary>4. On the Python edition's 30-feature data, raw k-means gave a much worse ARI than standardised; on this edition's 9-feature data it barely changed. Explain both.</summary>Euclidean distance is dominated by the features with the largest numerical range. In the 30-feature data one feature (<em>worst area</em>, in µm²) had values in the thousands and swamped the others, so the raw partition was effectively a threshold on area. The nine mlbench features are all cytological scores on the same 1–10 scale, so no feature can dominate, and standardising changes almost nothing. Standardising is still good practice because it costs nothing and protects against the first situation.</details>

<details><summary>5. Clusters \(A = \{(0,0), (0,1)\}\) and \(B = \{(3,0), (4,1)\}\). Compute \(D(A, B)\) under single, complete and average linkage.</summary>The four point distances are \(d((0,0),(3,0)) = 3\), \(d((0,0),(4,1)) = \sqrt{17} \approx 4.12\), \(d((0,1),(3,0)) = \sqrt{10} \approx 3.16\), \(d((0,1),(4,1)) = 4\). Single: \(3\). Complete: \(4.12\). Average: \((3 + 4.12 + 3.16 + 4)/4 \approx 3.57\).</details>

<details><summary>6. A dendrogram built with single linkage on patient data shows one cluster that absorbs almost everyone and a few singletons, however it is cut. What is happening and what would you try?</summary>Chaining: a few intermediate patients bridge the groups, and single linkage follows the bridge, merging the groups before they are complete, while isolated outliers remain as singletons. Try complete, average or Ward linkage, which require the whole clusters to be close; or remove outliers first; or use DBSCAN, which labels sparse points as noise rather than forcing them into clusters.</details>

<details><summary>7. Compute by hand the principal components of the covariance matrix \(\mathbf{S} = \begin{pmatrix} 3 & 1 \\ 1 & 3 \end{pmatrix}\) and the proportion of variance explained by the first.</summary>\(\det(\mathbf{S} - \lambda \mathbf{I}) = (3 - \lambda)^2 - 1 = 0\) gives \(\lambda_1 = 4\), \(\lambda_2 = 2\). For \(\lambda_1 = 4\): \(-w_1 + w_2 = 0\), so \(\mathbf{w}_1 = (1, 1)/\sqrt{2}\); for \(\lambda_2 = 2\), \(\mathbf{w}_2 = (1, -1)/\sqrt{2}\). Proportion explained by PC1: \(4 / 6 \approx 0.667\).</details>

<details><summary>8. On the nine-feature breast-cancer data the first standardised component explains about 66 % of the variance and loads almost equally on eight features. What does that tell you biologically?</summary>The eight cytological scores are strongly positively correlated: a malignant nucleus tends to be abnormal on all of them at once, so a single "degree of abnormality" axis accounts for two thirds of the variation. PC1 is essentially the average of those eight scores, and it is also the axis along which benign and malignant cases separate — which is why one principal component already classifies the cases at about 97 % (Figure 13).</details>

<details><summary>9. In the k-NN cross-validation of Section 4.6 the PCA was refitted in every fold even though PCA uses no labels. Why is the precaution necessary?</summary>Fitting PCA on all cases, including the test fold, lets the test cases influence the components (their means and directions), so the test fold is no longer independent of the preprocessing — leakage without labels. The effect is usually small for PCA but can be large for other unsupervised preprocessing (variance-based feature selection, imputation), and the principle is the same as in Supplement S1: every step that learns from data must see only the training fold.</details>

<details><summary>10. The first ICA component of an EEG recording is sometimes the eye-blink artefact and sometimes not. Why, and how is the artefact component identified in practice?</summary>ICA has a permutation ambiguity: the order of the components is arbitrary and changes with the initialisation. The artefact component is identified by its content — a waveform with large, slow deflections time-locked to blinks, a spatial pattern (the unmixing-matrix row) concentrated on the frontal electrodes, and, if available, a high correlation with the electro-oculogram channel — not by its position.</details>

<details><summary>11. Two recorded signals are mixtures of two Gaussian sources. Can ICA separate them? Can PCA? Explain.</summary>Neither. For Gaussian variables, uncorrelatedness is independence, and any rotation of two independent Gaussians yields two other independent Gaussians, so the sources are not identifiable: there is no non-Gaussianity for ICA to maximise. PCA returns uncorrelated components, but these are one arbitrary rotation among infinitely many equally "independent" ones. ICA requires that at most one source be Gaussian.</details>

<details><summary>12. An author clusters patients with k-means on a matrix \(X\), then reports that all variables of \(X\) differ significantly between clusters 1 and 2 \((p < 0.001)\) and concludes the clusters are real. What is wrong, and what would be acceptable evidence?</summary>The tests are circular: k-means chose the partition that maximises the separation of the clusters on exactly those columns of \(X\), so significant differences are guaranteed even in pure noise (Section 7.1). Acceptable evidence is external to the clustering variables: differences in a held-out variable or a later outcome, reproduction of the clusters in an independent cohort, stability under resampling, or an association with treatment response, as in the sepsis-phenotype study.</details>

## Glossary

<dl class="gloss">
<dt>Adjusted Rand index (ARI)</dt><dd>External index comparing two partitions by the pairs of cases they agree to group or separate, corrected for chance: 0 for a random partition, 1 for identical partitions; invariant to relabelling.</dd>
<dt>Agglomerative clustering</dt><dd>Bottom-up hierarchical clustering: start with one cluster per point and repeatedly merge the two closest clusters according to a linkage rule (R: <code>hclust</code>).</dd>
<dt>Blind source separation</dt><dd>Recovering unknown source signals from observed mixtures without knowing the mixing process; ICA is the main method.</dd>
<dt>Chaining</dt><dd>The tendency of single linkage to merge clusters through chains of intermediate points, producing elongated clusters.</dd>
<dt>Cophenetic distance / correlation</dt><dd>The height at which two points are first joined in a dendrogram / the correlation between cophenetic and original distances, measuring how faithfully the tree represents the data (R: <code>cophenetic</code>).</dd>
<dt>Covariance matrix</dt><dd>The \(p \times p\) symmetric matrix of feature variances (diagonal) and covariances (off-diagonal); its eigenvectors are the principal components.</dd>
<dt>Dendrogram</dt><dd>The tree produced by hierarchical clustering; leaves are cases, internal nodes are merges, and the height of a node is the distance at which the merge occurred.</dd>
<dt>Eigenvector / eigenvalue</dt><dd>A vector \(\mathbf{w}\) with \(\mathbf{S}\mathbf{w} = \lambda\mathbf{w}\); for a covariance matrix, a direction of variation and the variance along it (R: <code>eigen</code>).</dd>
<dt>Explained variance</dt><dd>The fraction \(\lambda_j / \sum_l \lambda_l\) of total variance carried by principal component \(j\).</dd>
<dt>FastICA</dt><dd>Fixed-point ICA algorithm that whitens the data and then finds directions of maximal non-Gaussianity one at a time; implemented from scratch in Section 5.3.</dd>
<dt>Gap statistic</dt><dd>Criterion for \(k\) comparing the observed \(\log J^\ast(k)\) with its expectation on structureless reference data; can select \(k = 1\).</dd>
<dt>Gaussian mixture model</dt><dd>Density model as a weighted sum of Gaussians, fitted by EM; its components are soft clusters with their own covariance (R: <code>mclust</code>).</dd>
<dt>Independent component analysis (ICA)</dt><dd>Linear decomposition into statistically independent components, found by maximising non-Gaussianity; determined only up to permutation, sign and scale.</dd>
<dt>k-means++</dt><dd>Initialisation that chooses successive centroids with probability proportional to their squared distance from the centroids already chosen.</dd>
<dt>Kurtosis / negentropy</dt><dd>Measures of non-Gaussianity: the standardised fourth moment minus 3 / the entropy deficit relative to a Gaussian of equal variance.</dd>
<dt>Linkage</dt><dd>The rule defining the distance between two clusters from the distances between their points: single (min), complete (max), average (mean), Ward (increase in within-cluster sum of squares).</dd>
<dt>Lloyd's algorithm</dt><dd>The alternating assignment/update procedure that minimises the k-means objective to a local minimum.</dd>
<dt>Loading</dt><dd>The weight of an original feature in a principal component (an entry of the eigenvector).</dd>
<dt>Medoid</dt><dd>The data point of a cluster with smallest total dissimilarity to the other members; the representative used by k-medoids/PAM (R: <code>cluster::pam</code>).</dd>
<dt>Principal component analysis (PCA)</dt><dd>Orthogonal linear transformation to uncorrelated components ordered by variance; equivalently, the subspace of minimum reconstruction error (R: <code>prcomp</code>).</dd>
<dt>Silhouette</dt><dd>For a point, \((b - a)/\max(a, b)\) where \(a\) is its mean distance to its own cluster and \(b\) to the nearest other; averaged over points, an internal index (R: <code>cluster::silhouette</code>).</dd>
<dt>Singular value decomposition (SVD)</dt><dd>Factorisation \(\mathbf{X} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^\top\); the right singular vectors are the principal components and \(\sigma_j^2/(n-1)\) the eigenvalues.</dd>
<dt>Unsupervised learning</dt><dd>Learning from a data matrix without labels: finding groups, low-dimensional summaries or a density.</dd>
<dt>Whitening</dt><dd>Transforming data to identity covariance (uncorrelated, unit-variance components); the first step of FastICA.</dd>
<dt>Within-cluster sum of squares (WCSS)</dt><dd>\(\sum_j \sum_{\mathbf{x}_i \in C_j} \lVert \mathbf{x}_i - \boldsymbol{\mu}_j \rVert^2\); the k-means objective (R: <code>kmeans()$tot.withinss</code>).</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">Street WN, Wolberg WH, Mangasarian OL. Nuclear feature extraction for breast tumor diagnosis. <em>Proc SPIE</em>. 1993;1905:861–870. <a href="https://doi.org/10.1117/12.148698">doi:10.1117/12.148698</a></li>
<li id="r2">Jain AK. Data clustering: 50 years beyond K-means. <em>Pattern Recognit Lett</em>. 2010;31(8):651–666. <a href="https://doi.org/10.1016/j.patrec.2009.09.011">doi:10.1016/j.patrec.2009.09.011</a></li>
<li id="r3">Xu R, Wunsch D. Survey of clustering algorithms. <em>IEEE Trans Neural Netw</em>. 2005;16(3):645–678. <a href="https://doi.org/10.1109/TNN.2005.845141">doi:10.1109/TNN.2005.845141</a></li>
<li id="r4">Leisch F, Dimitriadou E. mlbench: Machine Learning Benchmark Problems. R package. <a href="https://doi.org/10.32614/CRAN.package.mlbench">doi:10.32614/CRAN.package.mlbench</a></li>
<li id="r5">MacQueen J. Some methods for classification and analysis of multivariate observations. In: <em>Proc Fifth Berkeley Symp Math Stat Probab</em>, Vol. 1. University of California Press; 1967:281–297. <a href="https://projecteuclid.org/euclid.bsmsp/1200512992">projecteuclid.org/euclid.bsmsp/1200512992</a> (proceedings paper without DOI)</li>
<li id="r6">Lloyd S. Least squares quantization in PCM. <em>IEEE Trans Inf Theory</em>. 1982;28(2):129–137. <a href="https://doi.org/10.1109/TIT.1982.1056489">doi:10.1109/TIT.1982.1056489</a></li>
<li id="r7">Bezdek JC. <em>Pattern Recognition with Fuzzy Objective Function Algorithms</em>. New York: Plenum Press; 1981. <a href="https://doi.org/10.1007/978-1-4757-0450-1">doi:10.1007/978-1-4757-0450-1</a></li>
<li id="r8">Kaufman L, Rousseeuw PJ. <em>Finding Groups in Data: An Introduction to Cluster Analysis</em>. New York: Wiley; 1990. <a href="https://doi.org/10.1002/9780470316801">doi:10.1002/9780470316801</a></li>
<li id="r9">Sculley D. Web-scale k-means clustering. In: <em>Proc 19th Int Conf World Wide Web (WWW 2010)</em>. ACM; 2010:1177–1178. <a href="https://doi.org/10.1145/1772690.1772862">doi:10.1145/1772690.1772862</a></li>
<li id="r10">Arthur D, Vassilvitskii S. k-means++: the advantages of careful seeding. In: <em>Proc 18th Annual ACM-SIAM Symp Discrete Algorithms (SODA 2007)</em>. SIAM; 2007:1027–1035. <a href="https://dl.acm.org/doi/10.5555/1283383.1283494">dl.acm.org/doi/10.5555/1283383.1283494</a> (no CrossRef DOI)</li>
<li id="r11">Rousseeuw PJ. Silhouettes: a graphical aid to the interpretation and validation of cluster analysis. <em>J Comput Appl Math</em>. 1987;20:53–65. <a href="https://doi.org/10.1016/0377-0427(87)90125-7">doi:10.1016/0377-0427(87)90125-7</a></li>
<li id="r12">Tibshirani R, Walther G, Hastie T. Estimating the number of clusters in a data set via the gap statistic. <em>J R Stat Soc Ser B</em>. 2001;63(2):411–423. <a href="https://doi.org/10.1111/1467-9868.00293">doi:10.1111/1467-9868.00293</a></li>
<li id="r13">Hubert L, Arabie P. Comparing partitions. <em>J Classif</em>. 1985;2(1):193–218. <a href="https://doi.org/10.1007/BF01908075">doi:10.1007/BF01908075</a></li>
<li id="r14">Lance GN, Williams WT. A general theory of classificatory sorting strategies: 1. Hierarchical systems. <em>Comput J</em>. 1967;9(4):373–380. <a href="https://doi.org/10.1093/comjnl/9.4.373">doi:10.1093/comjnl/9.4.373</a></li>
<li id="r15">Ward JH. Hierarchical grouping to optimize an objective function. <em>J Am Stat Assoc</em>. 1963;58(301):236–244. <a href="https://doi.org/10.1080/01621459.1963.10500845">doi:10.1080/01621459.1963.10500845</a></li>
<li id="r16">Sokal RR, Rohlf FJ. The comparison of dendrograms by objective methods. <em>Taxon</em>. 1962;11(2):33–40. <a href="https://doi.org/10.2307/1217208">doi:10.2307/1217208</a></li>
<li id="r17">Eisen MB, Spellman PT, Brown PO, Botstein D. Cluster analysis and display of genome-wide expression patterns. <em>Proc Natl Acad Sci USA</em>. 1998;95(25):14863–14868. <a href="https://doi.org/10.1073/pnas.95.25.14863">doi:10.1073/pnas.95.25.14863</a></li>
<li id="r18">Aeberhard S, Forina M. Wine [dataset]. UCI Machine Learning Repository; 1991. <a href="https://doi.org/10.24432/C5PC7J">doi:10.24432/C5PC7J</a> (original of the samples × features table the Eisen-style heatmap of Section 3.6 simulates)</li>
<li id="r19">Hotelling H. Analysis of a complex of statistical variables into principal components. <em>J Educ Psychol</em>. 1933;24(6):417–441. <a href="https://doi.org/10.1037/h0071325">doi:10.1037/h0071325</a></li>
<li id="r20">Pearson K. On lines and planes of closest fit to systems of points in space. <em>Philos Mag</em>. 1901;2(11):559–572. <a href="https://doi.org/10.1080/14786440109462720">doi:10.1080/14786440109462720</a></li>
<li id="r21">Jolliffe IT, Cadima J. Principal component analysis: a review and recent developments. <em>Philos Trans R Soc A</em>. 2016;374(2065):20150202. <a href="https://doi.org/10.1098/rsta.2015.0202">doi:10.1098/rsta.2015.0202</a></li>
<li id="r22">Alpaydin E, Kaynak C. Optical Recognition of Handwritten Digits [dataset]. UCI Machine Learning Repository; 1998. <a href="https://doi.org/10.24432/C50P49">doi:10.24432/C50P49</a> (original of the image data the PCA visualisation of Section 4.6 simulates)</li>
<li id="r23">van der Maaten L, Hinton G. Visualizing data using t-SNE. <em>J Mach Learn Res</em>. 2008;9:2579–2605. <a href="https://www.jmlr.org/papers/v9/vandermaaten08a.html">jmlr.org/papers/v9/vandermaaten08a.html</a> (no DOI)</li>
<li id="r24">McInnes L, Healy J, Melville J. UMAP: uniform manifold approximation and projection for dimension reduction. <em>arXiv</em> 1802.03426; 2018. <a href="https://doi.org/10.48550/arXiv.1802.03426">doi:10.48550/arXiv.1802.03426</a></li>
<li id="r25">Comon P. Independent component analysis, a new concept? <em>Signal Process</em>. 1994;36(3):287–314. <a href="https://doi.org/10.1016/0165-1684(94)90029-9">doi:10.1016/0165-1684(94)90029-9</a></li>
<li id="r26">Bell AJ, Sejnowski TJ. An information-maximization approach to blind separation and blind deconvolution. <em>Neural Comput</em>. 1995;7(6):1129–1159. <a href="https://doi.org/10.1162/neco.1995.7.6.1129">doi:10.1162/neco.1995.7.6.1129</a></li>
<li id="r27">Hyvärinen A, Oja E. Independent component analysis: algorithms and applications. <em>Neural Netw</em>. 2000;13(4–5):411–430. <a href="https://doi.org/10.1016/S0893-6080(00)00026-5">doi:10.1016/S0893-6080(00)00026-5</a></li>
<li id="r28">Makeig S, Bell AJ, Jung T-P, Sejnowski TJ. Independent component analysis of electroencephalographic data. In: <em>Advances in Neural Information Processing Systems 8 (NIPS 1995)</em>. MIT Press; 1996:145–151. <a href="https://proceedings.neurips.cc/paper/1995">proceedings.neurips.cc/paper/1995</a> (no DOI)</li>
<li id="r29">Jung T-P, Makeig S, Humphries C, Lee T-W, McKeown MJ, Iragui V, Sejnowski TJ. Removing electroencephalographic artifacts by blind source separation. <em>Psychophysiology</em>. 2000;37(2):163–178. <a href="https://doi.org/10.1111/1469-8986.3720163">doi:10.1111/1469-8986.3720163</a></li>
<li id="r30">Dempster AP, Laird NM, Rubin DB. Maximum likelihood from incomplete data via the EM algorithm. <em>J R Stat Soc Ser B</em>. 1977;39(1):1–38. <a href="https://doi.org/10.1111/j.2517-6161.1977.tb01600.x">doi:10.1111/j.2517-6161.1977.tb01600.x</a></li>
<li id="r31">Ester M, Kriegel H-P, Sander J, Xu X. A density-based algorithm for discovering clusters in large spatial databases with noise. In: <em>Proc 2nd Int Conf Knowledge Discovery and Data Mining (KDD-96)</em>. AAAI Press; 1996:226–231. <a href="https://www.aaai.org/Papers/KDD/1996/KDD96-037.pdf">aaai.org/Papers/KDD/1996/KDD96-037.pdf</a> (no DOI)</li>
<li id="r32">Davies DL, Bouldin DW. A cluster separation measure. <em>IEEE Trans Pattern Anal Mach Intell</em>. 1979;PAMI-1(2):224–227. <a href="https://doi.org/10.1109/TPAMI.1979.4766909">doi:10.1109/TPAMI.1979.4766909</a></li>
<li id="r33">Seymour CW, Kennedy JN, Wang S, et al. Derivation, validation, and potential treatment implications of novel clinical phenotypes for sepsis. <em>JAMA</em>. 2019;321(20):2003–2017. <a href="https://doi.org/10.1001/jama.2019.5791">doi:10.1001/jama.2019.5791</a></li>
<li id="r34">Perou CM, Sørlie T, Eisen MB, et al. Molecular portraits of human breast tumours. <em>Nature</em>. 2000;406(6797):747–752. <a href="https://doi.org/10.1038/35021093">doi:10.1038/35021093</a></li>
<li id="r35">Golub TR, Slonim DK, Tamayo P, et al. Molecular classification of cancer: class discovery and class prediction by gene expression monitoring. <em>Science</em>. 1999;286(5439):531–537. <a href="https://doi.org/10.1126/science.286.5439.531">doi:10.1126/science.286.5439.531</a></li>
<li id="r36">Novembre J, Johnson T, Bryc K, et al. Genes mirror geography within Europe. <em>Nature</em>. 2008;456(7218):98–101. <a href="https://doi.org/10.1038/nature07331">doi:10.1038/nature07331</a></li>
<li id="r37">Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em> (the <code>MASS</code> package: <code>ginv</code>, <code>mvrnorm</code>). 4th ed. New York: Springer; 2002. <a href="https://doi.org/10.1007/978-0-387-21706-2">doi:10.1007/978-0-387-21706-2</a></li>
<li id="r38">Maechler M, Rousseeuw P, Struyf A, Hubert M, Hornik K. cluster: Cluster Analysis Basics and Extensions. R package. <a href="https://doi.org/10.32614/CRAN.package.cluster">doi:10.32614/CRAN.package.cluster</a></li>
</ol>

<h4>Textbooks &amp; resources</h4>
<ul>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021 — Chapter 12, PCA and clustering with R labs; the R companion to the whole course. Free PDF at statlearning.com.</li>
<li>Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2nd ed. Springer; 2009 — Chapter 14, "Unsupervised learning": cluster analysis, PCA, ICA, the gap statistic. Free PDF.</li>
<li>Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em> (MASS). 4th ed. Springer; 2002 — the reference for the R functions used here.</li>
<li>Hyvärinen A, Karhunen J, Oja E. <em>Independent Component Analysis</em>. Wiley; 2001 — the reference monograph on ICA.</li>
<li>Kassambara A. <em>Practical Guide to Cluster Analysis in R</em>. STHDA; 2017 — <code>factoextra</code>, <code>cluster</code> and <code>NbClust</code> for the methods of Sections 2–3.</li>
</ul>
